# KINETIA: entrenamiento del modelo de segmentación

U-Net con encoder MobileNetV3 Large preentrenado en ImageNet, 7 clases, entrada 320×320.

**Cómo usarlo:** activa la GPU T4 (*Entorno de ejecución → Cambiar tipo de entorno*) y pulsa *Entorno de ejecución → Ejecutar todo*. Si la sesión se corta, vuelve a pulsar *Ejecutar todo*: continúa sola desde la última época guardada en Drive. La guía completa está en `docs/COLAB_PASO_A_PASO.md`.

**Índice:** 1) Configuración · 2) Código del proyecto · 3) Dependencias y GPU · 4) Google Drive · 5) Datos · 6) Análisis exploratorio · 7) Preprocesamiento · 8) Entrenamiento y validación · 9) Curvas y sobreajuste/subajuste · 10) Exportación · 11) Evaluación · 12) Predicción sobre una imagen nueva · 13) Conclusiones · 14) Descarga.

**Ninguna cifra de este notebook está escrita a mano:** todas salen de ejecutar las celdas. Las estimaciones de tiempo se marcan como estimaciones.

## 1. Configuración

In [ ]:
import json, os, shutil, subprocess, sys, tempfile, time
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").strip().lower() in ("1", "true", "yes")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not SMOKE_TEST and not IN_COLAB:
    raise RuntimeError("Este notebook se entrena en Google Colab. Para una prueba local usa la variable de entorno SMOKE_TEST=1.")

# --- Parámetros del entrenamiento. Ajustados para la T4 gratis: AMP, 2 trabajadores.
EPOCHS = 2 if SMOKE_TEST else 40           # máximo; el early stopping suele parar antes
BATCH_SIZE = 4 if SMOKE_TEST else 32
NUM_WORKERS = 0 if SMOKE_TEST else 2       # Colab gratis tiene 2 vCPU
PATIENCE = 8                               # épocas sin mejorar el mIoU de validación
SMOKE_LIMITS = {"train": 16, "val": 6, "test": 6}  # solo se usan en SMOKE_TEST

if SMOKE_TEST:
    # Sin KINETIA_WORKDIR va a una carpeta temporal nueva, para no ensuciar el repositorio.
    WORK = Path(os.environ.get("KINETIA_WORKDIR") or tempfile.mkdtemp(prefix="kinetia_smoke_")).resolve()
else:
    WORK = Path("/content/kinetia")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
Path("ml/splits").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK / "ml"))

RAW_DIR = Path(os.environ.get("KINETIA_RAW_DIR", WORK / "raw")).resolve()
DATA_DIR = WORK / "data" / "processed"
OUT_DIR = WORK / "output"                 # los 5 archivos finales
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_VERSION = "smoke-test-no-oficial" if SMOKE_TEST else "seg-1.0.0"

print(f"Modo: {'SMOKE_TEST (prueba del flujo; sus cifras no son resultados)' if SMOKE_TEST else 'REAL'}")
print(f"Carpeta de trabajo: {WORK}")


def run(*cmd):
    """Ejecuta un script mostrando su salida en vivo; un error detiene el notebook."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "NO_ALBUMENTATIONS_UPDATE": "1"}
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", env=env, cwd=WORK)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"El comando falló con código {code}: {' '.join(cmd)}")

## 2. Código del proyecto

Cada celda escribe un archivo de `ml/`. El notebook no necesita ningún otro archivo.

In [ ]:
%%writefile ml/requirements.txt
# Entorno de ml/ (Python 3.11). En este portatil PyTorch es la version CPU:
#   pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cpu
#   pip install -r ml/requirements.txt
# En Colab NO se reinstalan torch, torchvision ni numpy: el notebook instala solo lo marcado colab.
torch==2.5.1
torchvision==0.20.1
numpy==2.1.3
opencv-python-headless==4.10.0.84  # colab
segmentation-models-pytorch==0.5.0  # colab
timm==1.0.12  # colab
# timm 1.0.12 no funciona con huggingface_hub 2.x: no encuentra los pesos preentrenados.
huggingface_hub==0.26.5  # colab
albumentations==1.4.24  # colab
onnx==1.17.0  # colab
onnxruntime==1.20.1  # colab
pyyaml==6.0.2
matplotlib==3.9.3
tqdm==4.67.1
pytest==8.3.4
nbconvert==7.16.4
nbformat==5.10.4
ipykernel==6.29.5
jupyter-client==8.6.3


In [ ]:
%%writefile ml/class_mapping.yaml
# Remapeo de COCO-Stuff 164k a las 7 clases de KINETIA (docs/CONTRATOS.md, seccion 1).
# Se define por NOMBRE de categoria, nunca por id numerico. Los ids solo aparecen en
# ml/coco_categories.json, que se extrajo de los JSON oficiales de COCO.
classes: [fondo, cielo, agua, vegetacion, fuego_luz, persona, animal]
ignore_index: 255

coco_stuff:
  # En stuffthingmaps_trainval2017.zip, valor del PNG = category_id - 1; 255 = unlabeled.
  # ml/count_pixels.py lo comprueba contra las anotaciones de instancias reales.
  png_value_offset: -1
  unlabeled_png_value: 255
  mapping:
    cielo: [sky-other, clouds]
    agua: [sea, river, water-other]
    vegetacion: [tree, grass, bush, plant-other, flower, leaves, moss]
    fuego_luz: [light]
    persona: [person]
    animal: [bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe]
  # Cualquier otra categoria etiquetada.
  default_class: fondo

# Fuente externa de fuego. Solo cuenta el fuego; todo lo demas queda en ignore_index.
# La mascara BoWFire marca el fuego en blanco y el resto en negro.
external_sources:
  bowfire:
    foreground_class: fuego_luz
    background: ignore
    foreground_threshold: 127


In [ ]:
%%writefile ml/coco_categories.json
{
"1": "person",
"2": "bicycle",
"3": "car",
"4": "motorcycle",
"5": "airplane",
"6": "bus",
"7": "train",
"8": "truck",
"9": "boat",
"10": "traffic light",
"11": "fire hydrant",
"13": "stop sign",
"14": "parking meter",
"15": "bench",
"16": "bird",
"17": "cat",
"18": "dog",
"19": "horse",
"20": "sheep",
"21": "cow",
"22": "elephant",
"23": "bear",
"24": "zebra",
"25": "giraffe",
"27": "backpack",
"28": "umbrella",
"31": "handbag",
"32": "tie",
"33": "suitcase",
"34": "frisbee",
"35": "skis",
"36": "snowboard",
"37": "sports ball",
"38": "kite",
"39": "baseball bat",
"40": "baseball glove",
"41": "skateboard",
"42": "surfboard",
"43": "tennis racket",
"44": "bottle",
"46": "wine glass",
"47": "cup",
"48": "fork",
"49": "knife",
"50": "spoon",
"51": "bowl",
"52": "banana",
"53": "apple",
"54": "sandwich",
"55": "orange",
"56": "broccoli",
"57": "carrot",
"58": "hot dog",
"59": "pizza",
"60": "donut",
"61": "cake",
"62": "chair",
"63": "couch",
"64": "potted plant",
"65": "bed",
"67": "dining table",
"70": "toilet",
"72": "tv",
"73": "laptop",
"74": "mouse",
"75": "remote",
"76": "keyboard",
"77": "cell phone",
"78": "microwave",
"79": "oven",
"80": "toaster",
"81": "sink",
"82": "refrigerator",
"84": "book",
"85": "clock",
"86": "vase",
"87": "scissors",
"88": "teddy bear",
"89": "hair drier",
"90": "toothbrush",
"92": "banner",
"93": "blanket",
"94": "branch",
"95": "bridge",
"96": "building-other",
"97": "bush",
"98": "cabinet",
"99": "cage",
"100": "cardboard",
"101": "carpet",
"102": "ceiling-other",
"103": "ceiling-tile",
"104": "cloth",
"105": "clothes",
"106": "clouds",
"107": "counter",
"108": "cupboard",
"109": "curtain",
"110": "desk-stuff",
"111": "dirt",
"112": "door-stuff",
"113": "fence",
"114": "floor-marble",
"115": "floor-other",
"116": "floor-stone",
"117": "floor-tile",
"118": "floor-wood",
"119": "flower",
"120": "fog",
"121": "food-other",
"122": "fruit",
"123": "furniture-other",
"124": "grass",
"125": "gravel",
"126": "ground-other",
"127": "hill",
"128": "house",
"129": "leaves",
"130": "light",
"131": "mat",
"132": "metal",
"133": "mirror-stuff",
"134": "moss",
"135": "mountain",
"136": "mud",
"137": "napkin",
"138": "net",
"139": "paper",
"140": "pavement",
"141": "pillow",
"142": "plant-other",
"143": "plastic",
"144": "platform",
"145": "playingfield",
"146": "railing",
"147": "railroad",
"148": "river",
"149": "road",
"150": "rock",
"151": "roof",
"152": "rug",
"153": "salad",
"154": "sand",
"155": "sea",
"156": "shelf",
"157": "sky-other",
"158": "skyscraper",
"159": "snow",
"160": "solid-other",
"161": "stairs",
"162": "stone",
"163": "straw",
"164": "structural-other",
"165": "table",
"166": "tent",
"167": "textile-other",
"168": "towel",
"169": "tree",
"170": "vegetable",
"171": "wall-brick",
"172": "wall-concrete",
"173": "wall-other",
"174": "wall-panel",
"175": "wall-stone",
"176": "wall-tile",
"177": "wall-wood",
"178": "water-other",
"179": "waterdrops",
"180": "window-blind",
"181": "window-other",
"182": "wood",
"183": "other"
}

In [ ]:
%%writefile ml/splits/bowfire_groups.json
[
 [
  "fire000",
  "fire002",
  "fire011",
  "fire012",
  "fire028",
  "fire089",
  "fire090",
  "fire091",
  "fire092",
  "fire093",
  "fire094",
  "fire095",
  "fire096",
  "fire097",
  "fire098",
  "fire099",
  "fire100",
  "fire101",
  "fire102",
  "fire103",
  "fire104",
  "fire105",
  "fire106",
  "fire107",
  "fire108",
  "fire109",
  "fire110",
  "fire111",
  "fire112",
  "fire113",
  "fire114",
  "fire117",
  "fire118"
 ],
 [
  "fire001"
 ],
 [
  "fire003",
  "fire022"
 ],
 [
  "fire004"
 ],
 [
  "fire005"
 ],
 [
  "fire006",
  "fire009"
 ],
 [
  "fire007"
 ],
 [
  "fire008"
 ],
 [
  "fire010"
 ],
 [
  "fire013",
  "fire036"
 ],
 [
  "fire014",
  "fire015"
 ],
 [
  "fire016"
 ],
 [
  "fire017",
  "fire018",
  "fire019",
  "fire021",
  "fire025",
  "fire026"
 ],
 [
  "fire020"
 ],
 [
  "fire023"
 ],
 [
  "fire024"
 ],
 [
  "fire027",
  "fire029",
  "fire068",
  "fire069",
  "fire073"
 ],
 [
  "fire030"
 ],
 [
  "fire031"
 ],
 [
  "fire032"
 ],
 [
  "fire033"
 ],
 [
  "fire034"
 ],
 [
  "fire035"
 ],
 [
  "fire037"
 ],
 [
  "fire038"
 ],
 [
  "fire039"
 ],
 [
  "fire040"
 ],
 [
  "fire041"
 ],
 [
  "fire042",
  "fire043"
 ],
 [
  "fire044"
 ],
 [
  "fire045",
  "fire046",
  "fire047",
  "fire048"
 ],
 [
  "fire049"
 ],
 [
  "fire050"
 ],
 [
  "fire051"
 ],
 [
  "fire052"
 ],
 [
  "fire053"
 ],
 [
  "fire054"
 ],
 [
  "fire055"
 ],
 [
  "fire056"
 ],
 [
  "fire057"
 ],
 [
  "fire058",
  "fire059"
 ],
 [
  "fire060",
  "fire061"
 ],
 [
  "fire062"
 ],
 [
  "fire063"
 ],
 [
  "fire064"
 ],
 [
  "fire065"
 ],
 [
  "fire066"
 ],
 [
  "fire067"
 ],
 [
  "fire070",
  "fire071",
  "fire072"
 ],
 [
  "fire074"
 ],
 [
  "fire075"
 ],
 [
  "fire076"
 ],
 [
  "fire077",
  "fire078"
 ],
 [
  "fire079"
 ],
 [
  "fire080"
 ],
 [
  "fire081"
 ],
 [
  "fire082"
 ],
 [
  "fire083"
 ],
 [
  "fire084"
 ],
 [
  "fire085"
 ],
 [
  "fire086"
 ],
 [
  "fire087"
 ],
 [
  "fire088"
 ],
 [
  "fire115"
 ],
 [
  "fire116"
 ]
]


In [ ]:
%%writefile ml/dataset.py
"""Dataset, remapeo de mascaras, preprocesado y utilidades de particion de KINETIA.

Fuente de verdad: docs/CONTRATOS.md, secciones 1, 2 y 3.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import tempfile
from pathlib import Path
from typing import Callable, Iterable, Optional, Sequence

import cv2
import numpy as np
import yaml

CLASSES = ["fondo", "cielo", "agua", "vegetacion", "fuego_luz", "persona", "animal"]
NUM_CLASSES = 7
IGNORE_INDEX = 255
FUEGO_LUZ = CLASSES.index("fuego_luz")
INPUT_SIZE = 320
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

SOURCE_COCO = "coco_stuff"
SOURCE_BOWFIRE = "bowfire"
SOURCES = (SOURCE_COCO, SOURCE_BOWFIRE)

# train.py solo puede usar estas particiones. Ver ForbiddenSplitError.
TRAINING_SPLITS = ("train", "val")
EVALUATION_SPLITS = ("test",)

ML_DIR = Path(__file__).resolve().parent
DEFAULT_MANIFEST = ML_DIR / "splits" / "manifest.csv"
DEFAULT_MAPPING = ML_DIR / "class_mapping.yaml"
DEFAULT_CATEGORIES = ML_DIR / "coco_categories.json"


class ForbiddenSplitError(RuntimeError):
    """Se intento usar una particion que este proceso no tiene permitida."""


class ManifestMismatchError(RuntimeError):
    """El manifiesto de datos no coincide con el hash esperado."""


# --------------------------------------------------------------------------
# Preprocesado (contrato, seccion 2.1). Debe coincidir con
# backend/app/services/preprocess.py::preprocess_seg con tolerancia 1e-5.
# --------------------------------------------------------------------------
def resize_image(image: np.ndarray) -> np.ndarray:
    """Resize directo a 320x320 con INTER_LINEAR, sin letterbox."""
    return cv2.resize(image, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR)


def resize_mask(mask: np.ndarray) -> np.ndarray:
    """Las mascaras usan INTER_NEAREST: interpolar indices de clase los corromperia."""
    return cv2.resize(mask, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_NEAREST)


def normalize_rgb(image_rgb: np.ndarray) -> np.ndarray:
    """uint8 RGB (H, W, 3) -> float32 CHW normalizado con la media y std de ImageNet."""
    x = image_rgb.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return np.transpose(x, (2, 0, 1))


def preprocess_seg(image_bgr: np.ndarray) -> np.ndarray:
    """BGR de OpenCV -> tensor float32 de forma [1, 3, 320, 320], listo para el ONNX."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return normalize_rgb(resize_image(rgb))[None, ...]


# --------------------------------------------------------------------------
# Remapeo de mascaras (siempre por nombre de categoria)
# --------------------------------------------------------------------------
def load_mapping(path: Path = DEFAULT_MAPPING) -> dict:
    with open(path, encoding="utf-8") as f:
        mapping = yaml.safe_load(f)
    if mapping["classes"] != CLASSES:
        raise ValueError(f"Las clases de {path} no coinciden con el contrato: {mapping['classes']}")
    return mapping


def load_categories(path: Path = DEFAULT_CATEGORIES) -> dict[int, str]:
    """id de categoria -> nombre, extraidos de los JSON oficiales de COCO."""
    with open(path, encoding="utf-8") as f:
        return {int(k): v for k, v in json.load(f).items()}


# Valor en la LUT para un valor de PNG que no corresponde a ninguna categoria conocida.
_UNKNOWN = 254


def build_coco_lut(mapping: dict, categories: dict[int, str]) -> np.ndarray:
    """Tabla de 256 entradas: valor del PNG de COCO-Stuff -> indice de clase."""
    cfg = mapping["coco_stuff"]
    by_name = {name: cid for cid, name in categories.items()}
    lut = np.full(256, _UNKNOWN, dtype=np.uint8)
    default = CLASSES.index(cfg["default_class"])
    for cid in categories:
        lut[cid + cfg["png_value_offset"]] = default
    for class_name, names in cfg["mapping"].items():
        for name in names:
            if name not in by_name:
                raise KeyError(f"La categoria '{name}' no existe en COCO-Stuff")
            lut[by_name[name] + cfg["png_value_offset"]] = CLASSES.index(class_name)
    lut[cfg["unlabeled_png_value"]] = IGNORE_INDEX
    return lut


def remap_coco_mask(raw: np.ndarray, lut: np.ndarray) -> np.ndarray:
    out = lut[raw]
    if (out == _UNKNOWN).any():
        bad = sorted(set(raw[out == _UNKNOWN].tolist()))
        raise ValueError(f"Valores de mascara sin categoria conocida: {bad}")
    return out


def remap_bowfire_mask(gt: np.ndarray, mapping: dict) -> np.ndarray:
    """Solo cuenta el fuego; todo lo demas queda en IGNORE_INDEX."""
    cfg = mapping["external_sources"][SOURCE_BOWFIRE]
    if gt.ndim == 3:
        gt = gt[..., 0]
    out = np.full(gt.shape, IGNORE_INDEX, dtype=np.uint8)
    out[gt > cfg["foreground_threshold"]] = CLASSES.index(cfg["foreground_class"])
    return out


# --------------------------------------------------------------------------
# Manifiesto de particion
# --------------------------------------------------------------------------
def manifest_sha256(path: Path = DEFAULT_MANIFEST) -> str:
    """SHA-256 del manifiesto con finales de linea normalizados a LF.

    git puede convertir el CSV a CRLF al clonar en Windows; sin normalizar, el mismo
    manifiesto daria un hash distinto en Colab y en el portatil.
    """
    data = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()


def read_manifest(path: Path = DEFAULT_MANIFEST) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if rows and list(rows[0].keys()) != ["image_id", "source", "split"]:
        raise ValueError(f"Columnas inesperadas en {path}: {list(rows[0].keys())}")
    return rows


def write_manifest(rows: Iterable[dict], path: Path = DEFAULT_MANIFEST) -> None:
    rows = sorted(rows, key=lambda r: (r["source"], r["image_id"]))
    text = "image_id,source,split\n" + "".join(
        f"{r['image_id']},{r['source']},{r['split']}\n" for r in rows
    )
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(text)


def rows_for_split(rows: Sequence[dict], split: str, allowed: Sequence[str]) -> list[dict]:
    """Filas de una particion; falla si el proceso no tiene permitida esa particion."""
    if split not in allowed:
        raise ForbiddenSplitError(f"La particion '{split}' no esta permitida aqui (permitidas: {tuple(allowed)})")
    return [r for r in rows if r["split"] == split]


def limit_rows(rows: Sequence[dict], limit: Optional[int]) -> list[dict]:
    """Subconjunto estable alternando fuentes, para que un limite incluya a todas."""
    if limit is None or limit >= len(rows):
        return list(rows)
    by_source: dict[str, list[dict]] = {}
    for row in sorted(rows, key=lambda r: (r["source"], r["image_id"])):
        by_source.setdefault(row["source"], []).append(row)
    out: list[dict] = []
    queues = [list(reversed(by_source[s])) for s in sorted(by_source)]
    while len(out) < limit:
        for queue in queues:
            if queue and len(out) < limit:
                out.append(queue.pop())
    return out


# --------------------------------------------------------------------------
# Lectura de disco
# --------------------------------------------------------------------------
def image_path(root: Path, row: dict) -> Path:
    return Path(root) / row["source"] / "images" / f"{row['image_id']}.png"


def mask_path(root: Path, row: dict) -> Path:
    return Path(root) / row["source"] / "masks" / f"{row['image_id']}.png"


def read_png(path: Path, flags: int) -> np.ndarray:
    # open() de Python y no cv2.imread: soporta rutas no ASCII en Windows y deja rastro
    # en los eventos de auditoria, que es como el test comprueba que train.py no abre test.
    with open(path, "rb") as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    image = cv2.imdecode(buf, flags)
    if image is None:
        raise IOError(f"No se pudo decodificar {path}")
    return image


def read_image_bgr(root: Path, row: dict) -> np.ndarray:
    return read_png(image_path(root, row), cv2.IMREAD_COLOR)


def read_mask(root: Path, row: dict) -> np.ndarray:
    return read_png(mask_path(root, row), cv2.IMREAD_GRAYSCALE)


class SegDataset:
    """Dataset map-style compatible con torch.utils.data.DataLoader.

    `allowed_splits` es obligatorio: quien construye el dataset declara que particiones
    puede leer, y cualquier otra fila provoca ForbiddenSplitError antes de abrir un archivo.
    """

    def __init__(
        self,
        root: Path,
        rows: Sequence[dict],
        allowed_splits: Sequence[str],
        transform: Optional[Callable] = None,
    ):
        for row in rows:
            if row["split"] not in allowed_splits:
                raise ForbiddenSplitError(
                    f"La imagen {row['image_id']} es de '{row['split']}' y solo se permite {tuple(allowed_splits)}"
                )
        self.root = Path(root)
        self.rows = list(rows)
        self.allowed_splits = tuple(allowed_splits)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[np.ndarray, np.ndarray]:
        row = self.rows[index]
        if row["split"] not in self.allowed_splits:  # las filas pueden mutar tras construir
            raise ForbiddenSplitError(f"Acceso prohibido a la particion '{row['split']}'")
        image_bgr = read_image_bgr(self.root, row)
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        rgb = resize_image(rgb)
        mask = resize_mask(read_mask(self.root, row))
        if self.transform is not None:
            augmented = self.transform(image=rgb, mask=mask)
            rgb, mask = augmented["image"], augmented["mask"]
        return normalize_rgb(rgb), mask.astype(np.int64)


# --------------------------------------------------------------------------
# Escritura atomica (Colab puede cortar en medio de una escritura)
# --------------------------------------------------------------------------
def atomic_write(path: Path, writer: Callable[[str], None]) -> None:
    """Escribe en un temporal de la misma carpeta y renombra.

    Si el proceso muere en medio de `writer`, el archivo anterior queda intacto.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        writer(tmp)
        with open(tmp, "rb+") as f:
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def atomic_write_json(path: Path, data: object) -> None:
    def writer(tmp: str) -> None:
        with open(tmp, "w", encoding="utf-8", newline="\n") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
            f.write("\n")

    atomic_write(path, writer)


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
%%writefile ml/seg_metrics.py
"""Metricas de segmentacion a partir de una matriz de confusion. Sin dependencias de PyTorch."""
from __future__ import annotations

import numpy as np

from dataset import CLASSES, IGNORE_INDEX, NUM_CLASSES


def confusion_update(matrix: np.ndarray, truth: np.ndarray, pred: np.ndarray) -> None:
    """Suma a `matrix` (filas = verdad, columnas = prediccion), omitiendo IGNORE_INDEX."""
    valid = truth != IGNORE_INDEX
    t = truth[valid].astype(np.int64)
    p = pred[valid].astype(np.int64)
    if t.size and (t.max() >= NUM_CLASSES or t.min() < 0):
        raise ValueError(f"Indice de clase invalido en la verdad: {np.unique(t).tolist()}")
    matrix += np.bincount(NUM_CLASSES * t + p, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_confusion(matrix: np.ndarray) -> dict:
    """mIoU y Dice se promedian sobre las clases con soporte o prediccion en estos datos."""
    matrix = np.asarray(matrix, dtype=np.int64)
    tp = np.diag(matrix).astype(np.float64)
    support = matrix.sum(axis=1).astype(np.float64)
    predicted = matrix.sum(axis=0).astype(np.float64)
    union = support + predicted - tp
    iou = np.divide(tp, union, out=np.zeros(NUM_CLASSES), where=union > 0)
    denom = support + predicted
    dice = np.divide(2 * tp, denom, out=np.zeros(NUM_CLASSES), where=denom > 0)
    present = union > 0
    return {
        "miou": float(iou[present].mean()) if present.any() else 0.0,
        "dice_macro": float(dice[present].mean()) if present.any() else 0.0,
        "pixel_accuracy": float(tp.sum() / matrix.sum()) if matrix.sum() else 0.0,
        "per_class": {
            name: {"iou": float(iou[i]), "dice": float(dice[i]), "support_pixels": int(support[i])}
            for i, name in enumerate(CLASSES)
        },
        "classes_without_data": [CLASSES[i] for i in np.where(~present)[0]],
    }


In [ ]:
%%writefile ml/model.py
"""Arquitectura del contrato: U-Net con encoder MobileNetV3 Large preentrenado en ImageNet."""
from __future__ import annotations

import segmentation_models_pytorch as smp
import torch

from dataset import NUM_CLASSES

ENCODER_NAME = "timm-mobilenetv3_large_100"


def build_model(pretrained: bool = True) -> torch.nn.Module:
    """La salida son logits: el softmax se aplica en la API (contrato, seccion 2.1)."""
    return smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights="imagenet" if pretrained else None,
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None,
    )


In [ ]:
%%writefile ml/prepare_data.py
"""Descarga y preprocesa las imagenes del manifiesto, y empaqueta/extrae el tar de Drive.

Las imagenes se guardan como PNG de 320x320 (resize directo INTER_LINEAR del contrato) y las
mascaras como PNG de 1 canal con indices 0-6 y 255 (INTER_NEAREST). Se usa PNG porque no
pierde informacion: el pixel en disco es exactamente la entrada que veria `preprocess_seg`.

Salida:
    <out>/coco_stuff/images/<id>.png, <out>/coco_stuff/masks/<id>.png
    <out>/bowfire/images/<id>.png,    <out>/bowfire/masks/<id>.png
"""
from __future__ import annotations

import argparse
import json
import os
import random
import tarfile
import time
import urllib.error
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Optional, Sequence

import cv2
import numpy as np

from dataset import (
    DEFAULT_MANIFEST,
    IGNORE_INDEX,
    NUM_CLASSES,
    SOURCE_BOWFIRE,
    SOURCE_COCO,
    atomic_write_json,
    build_coco_lut,
    image_path,
    limit_rows,
    load_categories,
    load_mapping,
    manifest_sha256,
    mask_path,
    read_manifest,
    read_mask,
    remap_bowfire_mask,
    remap_coco_mask,
    resize_image,
    resize_mask,
)

STUFF_ZIP_URL = "https://calvin.inf.ed.ac.uk/wp-content/uploads/data/cocostuffdataset/stuffthingmaps_trainval2017.zip"
STUFF_ZIP_BYTES = 659008564
BOWFIRE_ZIP_URL = "https://bitbucket.org/gbdi/bowfire-dataset/downloads/BoWFireDataset.zip"
BOWFIRE_ZIP_BYTES = 291275697
COCO_IMAGE_URL = "http://images.cocodataset.org/{subset}/{image_id}.jpg"
USER_AGENT = "kinetia-dataset-fetch/1.0"


def download(url: str, dst: Path, expected_bytes: Optional[int] = None, retries: int = 8) -> Path:
    """Descarga reanudable con Range. Si el archivo ya esta completo, no hace nada."""
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and (expected_bytes is None or dst.stat().st_size == expected_bytes):
        return dst
    part = dst.with_name(dst.name + ".part")
    for attempt in range(1, retries + 1):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": USER_AGENT}
        if start:
            headers["Range"] = f"bytes={start}-"
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as resp:
                mode = "ab" if start and resp.status == 206 else "wb"
                with open(part, mode) as f:
                    while True:
                        chunk = resp.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
            if expected_bytes is None or part.stat().st_size == expected_bytes:
                os.replace(part, dst)
                return dst
            print(f"Descarga incompleta de {dst.name}: {part.stat().st_size}/{expected_bytes} bytes; reintentando")
        except (urllib.error.URLError, OSError, TimeoutError) as exc:
            print(f"Intento {attempt}/{retries} de {dst.name} fallo: {exc}")
        time.sleep(min(30, 2**attempt))
    raise RuntimeError(f"No se pudo descargar {url}")


def _fetch_bytes(url: str, retries: int = 6) -> bytes:
    last: Exception = RuntimeError("sin intentos")
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=60) as resp:
                return resp.read()
        except (urllib.error.URLError, OSError, TimeoutError) as exc:
            last = exc
            time.sleep(min(20, 0.5 * 2**attempt) + random.random())
    raise RuntimeError(f"No se pudo descargar {url}: {last}")


def _write_png(path: Path, image: np.ndarray) -> None:
    ok, buf = cv2.imencode(".png", image)
    if not ok:
        raise IOError(f"No se pudo codificar {path}")
    tmp = path.with_name(path.name + f".{os.getpid()}.tmp")
    tmp.write_bytes(buf.tobytes())
    os.replace(tmp, path)


def _prepare_one(row: dict, out: Path, stuff: Optional[zipfile.ZipFile], index: dict, bow: Optional[zipfile.ZipFile], lut, mapping) -> None:
    img_out, mask_out = image_path(out, row), mask_path(out, row)
    if img_out.exists() and mask_out.exists():
        return
    image_id = row["image_id"]
    if row["source"] == SOURCE_COCO:
        member = index[image_id]  # p. ej. train2017/000000000139.png
        subset = member.split("/")[0]
        raw_image = _fetch_bytes(COCO_IMAGE_URL.format(subset=subset, image_id=image_id))
        raw_mask = stuff.read(member)
        mask = remap_coco_mask(cv2.imdecode(np.frombuffer(raw_mask, np.uint8), cv2.IMREAD_UNCHANGED), lut)
    elif row["source"] == SOURCE_BOWFIRE:
        raw_image = bow.read(index[f"bowfire/{image_id}"])
        raw_mask = bow.read(f"BoWFireDataset/dataset/gt/{image_id}_gt.png")
        mask = remap_bowfire_mask(cv2.imdecode(np.frombuffer(raw_mask, np.uint8), cv2.IMREAD_GRAYSCALE), mapping)
    else:
        raise ValueError(f"Fuente desconocida: {row['source']}")
    image = cv2.imdecode(np.frombuffer(raw_image, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise IOError(f"Imagen ilegible: {image_id}")
    if image.shape[:2] != mask.shape[:2]:
        raise ValueError(f"Imagen y mascara de {image_id} con tamanos distintos: {image.shape[:2]} {mask.shape[:2]}")
    _write_png(img_out, resize_image(image))
    _write_png(mask_out, resize_mask(mask))


def prepare_rows(rows: Sequence[dict], out: Path, raw_dir: Path, threads: int = 16) -> dict:
    """Prepara las filas dadas. Reanuda sola: lo que ya existe no se vuelve a bajar."""
    out, raw_dir = Path(out), Path(raw_dir)
    for source in (SOURCE_COCO, SOURCE_BOWFIRE):
        (out / source / "images").mkdir(parents=True, exist_ok=True)
        (out / source / "masks").mkdir(parents=True, exist_ok=True)
    pending = [r for r in rows if not (image_path(out, r).exists() and mask_path(out, r).exists())]
    print(f"Imagenes: {len(rows)} en total, {len(pending)} por preparar", flush=True)
    if not pending:
        return {"prepared": 0, "failed": 0, "seconds": 0.0}

    mapping = load_mapping()
    lut = build_coco_lut(mapping, load_categories())
    index: dict[str, str] = {}
    stuff = bow = None
    if any(r["source"] == SOURCE_COCO for r in pending):
        stuff = zipfile.ZipFile(download(STUFF_ZIP_URL, raw_dir / "stuffthingmaps_trainval2017.zip", STUFF_ZIP_BYTES))
        for name in stuff.namelist():
            if name.endswith(".png"):
                index[Path(name).stem] = name
    if any(r["source"] == SOURCE_BOWFIRE for r in pending):
        bow = zipfile.ZipFile(download(BOWFIRE_ZIP_URL, raw_dir / "BoWFireDataset.zip", BOWFIRE_ZIP_BYTES))
        for name in bow.namelist():
            if "/dataset/img/" in name and "__MACOSX" not in name and not name.endswith("/"):
                index[f"bowfire/{Path(name).stem}"] = name

    start = time.time()
    failed: list[tuple[str, str]] = []
    done = 0
    with ThreadPoolExecutor(threads) as pool:
        futures = {pool.submit(_prepare_one, r, out, stuff, index, bow, lut, mapping): r for r in pending}
        for future in as_completed(futures):
            row = futures[future]
            try:
                future.result()
            except Exception as exc:  # se reporta al final: una imagen no debe tumbar 15 000
                failed.append((row["image_id"], repr(exc)))
            done += 1
            if done % 500 == 0 or done == len(pending):
                rate = done / max(time.time() - start, 1e-6)
                print(f"  {done}/{len(pending)} ({rate:.0f} img/s medidos; faltan ~{(len(pending) - done) / rate / 60:.1f} min)", flush=True)
    if failed:
        raise RuntimeError(f"{len(failed)} imagenes fallaron; vuelve a ejecutar para reintentar. Primeras: {failed[:5]}")
    return {"prepared": len(pending), "failed": 0, "seconds": time.time() - start}


def make_tar(out: Path, rows: Sequence[dict], tar_path: Path) -> dict:
    """Tar sin comprimir (los PNG ya lo estan). Se escribe en temporal y se renombra."""
    out, tar_path = Path(out), Path(tar_path)
    tar_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = tar_path.with_name(tar_path.name + ".tmp")
    with tarfile.open(tmp, "w") as tar:
        for row in sorted(rows, key=lambda r: (r["source"], r["image_id"])):
            for path in (image_path(out, row), mask_path(out, row)):
                tar.add(path, arcname=path.relative_to(out).as_posix())
    os.replace(tmp, tar_path)
    return {"bytes": tar_path.stat().st_size, "num_images": len(rows)}


def extract_tar(tar_path: Path, out: Path) -> None:
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r") as tar:
        members = tar.getmembers()
        for member in members:  # el tar lo escribimos nosotros, pero se valida igual
            target = (out / member.name).resolve()
            if not str(target).startswith(str(out.resolve())) or not member.isfile():
                raise ValueError(f"Entrada sospechosa en el tar: {member.name}")
        tar.extractall(out, members=members)


def compute_class_weights(root: Path, train_rows: Sequence[dict]) -> dict:
    """Peso por clase a partir de los pixeles de TRAIN (raiz de la frecuencia mediana)."""
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for row in train_rows:
        hist = np.bincount(read_mask(root, row).ravel(), minlength=256)
        counts += hist[:NUM_CLASSES]
    freq = counts / max(counts.sum(), 1)
    present = freq[freq > 0]
    median = np.median(present) if present.size else 1.0
    weights = np.where(freq > 0, np.sqrt(median / np.maximum(freq, 1e-12)), 1.0)
    weights = np.clip(weights, 0.5, 8.0)
    return {"pixels": counts.tolist(), "weights": [float(w) for w in weights]}


def check_prepared(rows: Sequence[dict], root: Path) -> list[str]:
    return [
        r["image_id"]
        for r in rows
        if not (image_path(root, r).exists() and mask_path(root, r).exists())
    ]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=Path("data/processed"))
    parser.add_argument("--raw-dir", type=Path, default=Path("data/raw"))
    parser.add_argument("--manifest", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--splits", default="train,val,test")
    parser.add_argument("--limit-per-split", type=int, default=None)
    parser.add_argument("--threads", type=int, default=16)
    parser.add_argument("--tar", type=Path, default=None, help="Ademas, empaqueta en este tar")
    args = parser.parse_args()

    manifest = read_manifest(args.manifest)
    rows: list[dict] = []
    for split in args.splits.split(","):
        rows += limit_rows([r for r in manifest if r["split"] == split], args.limit_per_split)
    summary = prepare_rows(rows, args.out, args.raw_dir, args.threads)
    missing = check_prepared(rows, args.out)
    if missing:
        raise RuntimeError(f"Faltan {len(missing)} imagenes: {missing[:5]}")
    print(json.dumps(summary))
    total_bytes = sum(image_path(args.out, r).stat().st_size + mask_path(args.out, r).stat().st_size for r in rows)
    print(f"Tamano en disco de {len(rows)} imagenes + mascaras: {total_bytes / 1e9:.3f} GB (medido)")
    if args.tar:
        info = make_tar(args.out, rows, args.tar)
        info["manifest_sha256"] = manifest_sha256(args.manifest)
        atomic_write_json(args.tar.with_suffix(".json"), info)
        print(f"Tar {args.tar}: {info['bytes'] / 1e9:.3f} GB (medido)")
    assert IGNORE_INDEX == 255


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/make_splits.py
"""Selecciona ~15 000 imagenes de COCO-Stuff y genera ml/splits/manifest.csv (70/15/15, seed=42).

Por que no se usan las particiones oficiales de COCO:
  1. El conjunto `test2017` oficial no tiene mascaras publicas, asi que no sirve para medir.
  2. Solo se usa un subconjunto reforzado en clases escasas; train2017 y val2017 se tratan
     como una sola bolsa y se reparten de nuevo con semilla fija. Asi las 3 particiones
     tienen la misma distribucion de clases y BoWFire puede repartirse en las tres.

El manifiesto se genera SOLO con anotaciones: no hace falta descargar imagenes de COCO.

Uso:
    python ml/make_splits.py --stuff-zip data/raw/stuffthingmaps_trainval2017.zip \
                             --bowfire-zip data/_fire_candidates/bowfire/BoWFireDataset.zip
"""
from __future__ import annotations

import argparse
import io
import json
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np

from dataset import (
    CLASSES,
    DEFAULT_MANIFEST,
    FUEGO_LUZ,
    NUM_CLASSES,
    SOURCE_BOWFIRE,
    SOURCE_COCO,
    build_coco_lut,
    load_categories,
    load_mapping,
    write_manifest,
)

SEED = 42
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
BOWFIRE_GROUPS = Path(__file__).resolve().parent / "splits" / "bowfire_groups.json"

# Imagenes pedidas para cada clase escasa y area minima (fraccion de la imagen) para contar.
# Se piden en orden de escasez. Las imagenes que ya entraron por otra clase cuentan para esta.
QUOTAS = [
    ("fuego_luz", 4000, 0.001),
    ("agua", 3000, 0.02),
    ("animal", 3500, 0.01),
    ("cielo", 2500, 0.05),
    ("vegetacion", 2500, 0.05),
    ("persona", 2500, 0.02),
]
HIST_SIZE = 256


def _hist_worker(args: tuple[str, list[str]]) -> tuple[list[str], np.ndarray]:
    zip_path, names = args
    out = np.zeros((len(names), HIST_SIZE), dtype=np.int64)
    with zipfile.ZipFile(zip_path) as zf:
        for i, name in enumerate(names):
            raw = cv2.imdecode(np.frombuffer(zf.read(name), np.uint8), cv2.IMREAD_UNCHANGED)
            out[i] = np.bincount(raw.ravel(), minlength=HIST_SIZE)
    return names, out


def compute_histograms(zip_path: Path, cache: Path, workers: int = 8) -> tuple[np.ndarray, np.ndarray]:
    """Histograma de valores de PNG por imagen. Devuelve (nombres, hist) y usa cache."""
    if cache.exists():
        data = np.load(cache)
        return data["names"], data["hist"]
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".png") and "__MACOSX" not in n)
    print(f"Leyendo {len(names)} mascaras reales de {zip_path} ...", flush=True)
    chunks = [names[i : i + 2000] for i in range(0, len(names), 2000)]
    out_names: list[str] = []
    out_hist: list[np.ndarray] = []
    with ProcessPoolExecutor(workers) as pool:
        for i, (chunk_names, chunk_hist) in enumerate(pool.map(_hist_worker, [(str(zip_path), c) for c in chunks])):
            out_names += chunk_names
            out_hist.append(chunk_hist)
            print(f"  {min((i + 1) * 2000, len(names))}/{len(names)}", flush=True)
    hist = np.concatenate(out_hist)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache, names=np.array(out_names), hist=hist)
    return np.array(out_names), hist


def class_histograms(hist: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(pixeles por clase 0..6, pixeles unlabeled) de cada imagen, por remapeo real."""
    lut = build_coco_lut(load_mapping(), load_categories())
    unknown = np.where((lut == 254) & (hist.sum(axis=0) > 0))[0]
    if unknown.size:
        raise ValueError(f"Valores de PNG sin categoria en COCO-Stuff: {unknown.tolist()}")
    per_class = np.zeros((hist.shape[0], NUM_CLASSES), dtype=np.int64)
    for value in range(HIST_SIZE):
        if lut[value] < NUM_CLASSES:
            per_class[:, lut[value]] += hist[:, value]
    return per_class, hist[:, 255]


def select_images(per_class: np.ndarray, n_target: int, rng: np.random.Generator) -> np.ndarray:
    """Cuotas por clase escasa y resto al azar. Devuelve indices de las imagenes elegidas."""
    frac = per_class / np.maximum(per_class.sum(axis=1, keepdims=True), 1)
    chosen = np.zeros(len(per_class), dtype=bool)
    for name, quota, min_frac in QUOTAS:
        c = CLASSES.index(name)
        eligible = frac[:, c] >= min_frac
        missing = quota - int((chosen & eligible).sum())
        pool = np.where(eligible & ~chosen)[0]
        if missing > 0:
            take = rng.permutation(pool)[: min(missing, n_target - int(chosen.sum()))]
            chosen[take] = True
        print(f"  cuota {name}: {int(eligible.sum())} elegibles, {int((chosen & eligible).sum())} incluidas")
    remaining = n_target - int(chosen.sum())
    if remaining > 0:
        pool = np.where(~chosen & (per_class.sum(axis=1) > 0))[0]
        chosen[rng.permutation(pool)[:remaining]] = True
    return np.where(chosen)[0]


def stratum(row_hist: np.ndarray) -> int:
    """Clase mas escasa presente en la imagen, para repartir las escasas en las 3 particiones."""
    frac = row_hist / max(row_hist.sum(), 1)
    for name, _, min_frac in QUOTAS:
        c = CLASSES.index(name)
        if frac[c] >= min_frac:
            return c
    return 0


def split_groups(groups: list[list[str]], rng: np.random.Generator) -> dict[str, str]:
    """Reparte grupos enteros buscando 70/15/15 de imagenes."""
    total = sum(len(g) for g in groups)
    targets = {k: v * total for k, v in RATIOS.items()}
    counts = {k: 0 for k in RATIOS}
    out: dict[str, str] = {}
    order = rng.permutation(len(groups))
    # Primero los grupos grandes: son los dificiles de encajar.
    for i in sorted(order, key=lambda j: -len(groups[j])):
        group = groups[i]
        split = max(RATIOS, key=lambda k: (targets[k] - counts[k]) / targets[k])
        for item in group:
            out[item] = split
        counts[split] += len(group)
    return out


def split_coco(ids: list[str], strata: list[int], rng: np.random.Generator) -> dict[str, str]:
    out: dict[str, str] = {}
    strata_arr = np.array(strata)
    for value in sorted(set(strata)):
        members = np.where(strata_arr == value)[0]
        members = rng.permutation(members)
        n_train = int(round(len(members) * RATIOS["train"]))
        n_val = int(round(len(members) * RATIOS["val"]))
        for pos, idx in enumerate(members):
            out[ids[idx]] = "train" if pos < n_train else "val" if pos < n_train + n_val else "test"
    return out


def bowfire_fire_ids(zip_path: Path) -> list[str]:
    """Imagenes con fuego. Las `not*` no tienen fuego y sin el quedarian enteras en ignore."""
    with zipfile.ZipFile(zip_path) as zf:
        names = [
            n
            for n in zf.namelist()
            if "/dataset/img/" in n and "__MACOSX" not in n and Path(n).name.startswith("fire") and n.endswith((".png", ".jpg"))
        ]
    return sorted(Path(n).stem for n in names)


def compute_bowfire_groups(zip_path: Path, max_distance: int = 12) -> list[list[str]]:
    """Agrupa casi-duplicados con dHash de 64 bits.

    Muchas fotos de BoWFire son rafagas de la misma escena. Si caen en train y test, el
    conjunto de prueba mediria memoria y no generalizacion, asi que van juntas.
    """
    ids = bowfire_fire_ids(zip_path)
    hashes = []
    with zipfile.ZipFile(zip_path) as zf:
        by_stem = {
            Path(n).stem: n for n in zf.namelist() if "/dataset/img/" in n and "__MACOSX" not in n and not n.endswith("/")
        }
        for image_id in ids:
            img = cv2.imdecode(np.frombuffer(io.BytesIO(zf.read(by_stem[image_id])).getvalue(), np.uint8), cv2.IMREAD_GRAYSCALE)
            small = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
            hashes.append((small[:, 1:] > small[:, :-1]).ravel())
    parent = list(range(len(ids)))

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            if int((hashes[i] != hashes[j]).sum()) <= max_distance:
                parent[find(i)] = find(j)
    groups: dict[int, list[str]] = {}
    for i, image_id in enumerate(ids):
        groups.setdefault(find(i), []).append(image_id)
    return sorted(groups.values(), key=lambda g: g[0])


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stuff-zip", type=Path, required=True)
    parser.add_argument("--bowfire-zip", type=Path, required=True)
    parser.add_argument("--n-coco", type=int, default=15000)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--hist-cache", type=Path, default=Path("data/raw/coco_png_histograms.npz"))
    parser.add_argument("--out", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--workers", type=int, default=8)
    args = parser.parse_args()

    names, hist = compute_histograms(args.stuff_zip, args.hist_cache, args.workers)
    per_class, _ = class_histograms(hist)
    ids = [Path(str(n)).stem for n in names]

    rng = np.random.default_rng(args.seed)
    print(f"Imagenes en COCO-Stuff (train2017 + val2017): {len(ids)}")
    selected = select_images(per_class, args.n_coco, rng)
    selected_ids = [ids[i] for i in selected]
    coco_split = split_coco(selected_ids, [stratum(per_class[i]) for i in selected], rng)

    if BOWFIRE_GROUPS.exists():
        groups = json.loads(BOWFIRE_GROUPS.read_text(encoding="utf-8"))
    else:
        groups = compute_bowfire_groups(args.bowfire_zip)
        BOWFIRE_GROUPS.write_text(json.dumps(groups, indent=1) + "\n", encoding="utf-8", newline="\n")
    fire_split = split_groups(groups, rng)

    rows = [{"image_id": k, "source": SOURCE_COCO, "split": v} for k, v in coco_split.items()]
    rows += [{"image_id": k, "source": SOURCE_BOWFIRE, "split": v} for k, v in fire_split.items()]
    write_manifest(rows, args.out)

    for source in (SOURCE_COCO, SOURCE_BOWFIRE):
        sub = [r for r in rows if r["source"] == source]
        counts = {s: sum(r["split"] == s for r in sub) for s in RATIOS}
        print(source, len(sub), {k: f"{v} ({v / len(sub):.1%})" for k, v in counts.items()})
    print(f"Fuego en COCO (clase {FUEGO_LUZ}) solo significa `light`: ver ml/reports/class_pixel_counts.json")
    print(f"Escrito {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/count_pixels.py
"""Conteo real de pixeles por clase tras el remapeo. Escribe ml/reports/class_pixel_counts.json.

Sale de leer cada mascara de COCO-Stuff y BoWFire a su resolucion nativa, sin estimaciones.
Tambien busca una categoria de fuego en la lista real de categorias y comprueba que
valor de PNG = category_id - 1 comparando con las anotaciones de instancias de COCO.

Uso:
    python ml/count_pixels.py --stuff-zip data/raw/stuffthingmaps_trainval2017.zip \
                              --bowfire-zip data/_fire_candidates/bowfire/BoWFireDataset.zip
"""
from __future__ import annotations

import argparse
import io
import json
import re
import urllib.request
import zipfile
from pathlib import Path

import cv2
import numpy as np

from dataset import (
    CLASSES,
    IGNORE_INDEX,
    SOURCE_BOWFIRE,
    SOURCE_COCO,
    atomic_write_json,
    load_categories,
    load_mapping,
    manifest_sha256,
    read_manifest,
    remap_bowfire_mask,
)
from make_splits import class_histograms, compute_histograms

FIRE_PATTERN = re.compile(r"fire|flame|burn|blaze|ember|smoke|candle|torch", re.I)
INSTANCES_ZIP = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"


class _HttpFile(io.RawIOBase):
    """Archivo de solo lectura sobre HTTP Range, para leer un JSON de un zip enorme."""

    def __init__(self, url: str):
        self.url, self.pos = url, 0
        head = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        self.size = int(head.headers["Content-Length"])

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def readinto(self, buf) -> int:
        if self.pos >= self.size:
            return 0
        end = min(self.pos + len(buf), self.size) - 1
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        data = urllib.request.urlopen(req, timeout=120).read()
        buf[: len(data)] = data
        self.pos += len(data)
        return len(data)


def verify_png_offset(names: np.ndarray, hist: np.ndarray) -> dict:
    """Compara el area de `person` en el PNG con el area de las instancias reales de val2017."""
    with zipfile.ZipFile(io.BufferedReader(_HttpFile(INSTANCES_ZIP), buffer_size=1 << 20)) as zf:
        data = json.loads(zf.read("annotations/instances_val2017.json"))
    categories = {c["name"]: c["id"] for c in data["categories"]}
    index = {int(Path(str(n)).stem): i for i, n in enumerate(names) if "val2017" in str(n)}
    results = {}
    for name in ("person", "dog", "airplane", "toilet"):
        cid = categories[name]
        per_image: dict[int, float] = {}
        for ann in data["annotations"]:
            if ann["category_id"] == cid and not ann["iscrowd"]:
                per_image[ann["image_id"]] = per_image.get(ann["image_id"], 0.0) + ann["area"]
        ids = [i for i in per_image if i in index]
        polygon_area = np.array([per_image[i] for i in ids])
        png_area = np.array([hist[index[i], cid - 1] for i in ids], dtype=np.float64)
        results[name] = {
            "category_id": cid,
            "images": len(ids),
            "correlation_png_value_id_minus_1": float(np.corrcoef(polygon_area, png_area)[0, 1]),
            "total_area_ratio_png_over_annotations": float(png_area.sum() / polygon_area.sum()),
        }
    return results


def table(per_class: np.ndarray, unlabeled: int) -> dict:
    labeled = int(per_class.sum())
    total = labeled + int(unlabeled)
    return {
        "per_class": {
            name: {
                "pixels": int(per_class[i]),
                "pct_of_labeled": round(100.0 * int(per_class[i]) / labeled, 4) if labeled else 0.0,
            }
            for i, name in enumerate(CLASSES)
        },
        "ignore_255_pixels": int(unlabeled),
        "ignore_pct_of_total": round(100.0 * int(unlabeled) / total, 4) if total else 0.0,
    }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stuff-zip", type=Path, required=True)
    parser.add_argument("--bowfire-zip", type=Path, required=True)
    parser.add_argument("--hist-cache", type=Path, default=Path("data/raw/coco_png_histograms.npz"))
    parser.add_argument("--out", type=Path, default=Path(__file__).resolve().parent / "reports" / "class_pixel_counts.json")
    parser.add_argument("--skip-offset-check", action="store_true")
    args = parser.parse_args()

    categories = load_categories()
    names, hist = compute_histograms(args.stuff_zip, args.hist_cache)
    per_class, unlabeled = class_histograms(hist)
    ids = {Path(str(n)).stem: i for i, n in enumerate(names)}

    raw_totals = hist.sum(axis=0)
    fire_matches = {name: int(raw_totals[cid - 1]) for cid, name in categories.items() if FIRE_PATTERN.search(name)}
    light_id = next(c for c, n in categories.items() if n == "light")

    manifest = read_manifest()
    report = {
        "generated_by": "ml/count_pixels.py",
        "resolution": "nativa de cada mascara",
        "dataset_manifest_sha256": manifest_sha256(),
        "fire_category_check": {
            "categories_total": len(categories),
            "pattern_searched": FIRE_PATTERN.pattern,
            "categories_matching_with_pixels": fire_matches,
            "has_fire_category": any(name != "fire hydrant" for name in fire_matches),
            "note": "`fire hydrant` es un hidrante. COCO-Stuff no tiene categoria de fuego.",
            "light_category_id": light_id,
            "light_pixels": int(raw_totals[light_id - 1]),
            "images_with_light": int((hist[:, light_id - 1] > 0).sum()),
        },
        "all_coco_stuff_trainval": {"num_images": len(names), **table(per_class.sum(axis=0), unlabeled.sum())},
    }

    mapping = load_mapping()
    fire_px: dict[str, np.ndarray] = {}
    with zipfile.ZipFile(args.bowfire_zip) as zf:
        for row in manifest:
            if row["source"] != SOURCE_BOWFIRE:
                continue
            gt = cv2.imdecode(
                np.frombuffer(zf.read(f"BoWFireDataset/dataset/gt/{row['image_id']}_gt.png"), np.uint8),
                cv2.IMREAD_GRAYSCALE,
            )
            remapped = remap_bowfire_mask(gt, mapping)
            fire_px[row["image_id"]] = np.bincount(remapped.ravel(), minlength=256)

    for split in ("all", "train", "val", "test"):
        rows = [r for r in manifest if split == "all" or r["split"] == split]
        coco_rows = [ids[r["image_id"]] for r in rows if r["source"] == SOURCE_COCO]
        fire_rows = [fire_px[r["image_id"]] for r in rows if r["source"] == SOURCE_BOWFIRE]
        counts = per_class[coco_rows].sum(axis=0)
        ignored = int(unlabeled[coco_rows].sum())
        fire_only = np.zeros(len(CLASSES), dtype=np.int64)
        for h in fire_rows:
            fire_only += h[: len(CLASSES)]
            ignored += int(h[IGNORE_INDEX])
        report[f"selected_{split}"] = {
            "num_images": {SOURCE_COCO: len(coco_rows), SOURCE_BOWFIRE: len(fire_rows)},
            "fuego_luz_from_coco_light": int(counts[CLASSES.index("fuego_luz")]),
            "fuego_luz_from_bowfire_fire": int(fire_only[CLASSES.index("fuego_luz")]),
            **table(counts + fire_only, ignored),
        }

    if not args.skip_offset_check:
        report["png_value_equals_category_id_minus_1_check"] = verify_png_offset(names, hist)

    atomic_write_json(args.out, report)
    print(json.dumps({k: report[k] for k in report if k != "dataset_manifest_sha256"}, indent=1, ensure_ascii=False))
    print(f"Escrito {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/train.py
"""Entrenamiento de la segmentacion de KINETIA, reanudable por epoca.

Reanudacion: en cada epoca se guardan modelo, optimizador, scheduler, scaler, estado de RNG,
mejor mIoU, contador de early stopping e historial. Al arrancar, si existe `last.ckpt` se
continua sola desde la epoca siguiente; una epoca cortada a medias se repite desde su inicio.

Este script NO puede leer la particion de prueba: solo pide filas de `TRAINING_SPLITS` y el
dataset rechaza cualquier otra. Hay un test automatizado que lo comprueba.
"""
from __future__ import annotations

import argparse
import copy
import json
import random
import shutil
import time
from pathlib import Path
from typing import Optional

import albumentations as A
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from dataset import (
    CLASSES,
    DEFAULT_MANIFEST,
    IGNORE_INDEX,
    NUM_CLASSES,
    SOURCE_BOWFIRE,
    TRAINING_SPLITS,
    ManifestMismatchError,
    SegDataset,
    atomic_write,
    atomic_write_json,
    limit_rows,
    manifest_sha256,
    read_manifest,
    rows_for_split,
)
from model import ENCODER_NAME, build_model
from prepare_data import compute_class_weights
from seg_metrics import metrics_from_confusion

LAST_CKPT = "last.ckpt"
PREV_CKPT = "last.prev.ckpt"
BEST_MODEL = "kinetia_seg.pt"
HISTORY = "history.json"
CLASS_WEIGHTS = "class_weights.json"
# BoWFire tiene pocas imagenes frente a COCO: se ven mas veces por epoca.
BOWFIRE_SAMPLE_FACTOR = 8.0


def build_train_transform() -> A.Compose:
    """Aumentos. La salida sigue siendo 320x320; los pixeles rellenados quedan en IGNORE_INDEX.

    El tono se toca poco para que el fuego no deje de parecer fuego.
    """
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.Affine(
                scale=(0.85, 1.25),
                translate_percent=(-0.05, 0.05),
                rotate=(-10, 10),
                interpolation=cv2.INTER_LINEAR,
                mask_interpolation=cv2.INTER_NEAREST,
                cval=0,
                cval_mask=IGNORE_INDEX,
                mode=cv2.BORDER_CONSTANT,
                p=0.7,
            ),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=6, sat_shift_limit=20, val_shift_limit=15, p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.1),
        ]
    )


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)


def _worker_init(worker_id: int) -> None:
    # La semilla base sale del RNG principal, que se restaura al reanudar.
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)
    info = torch.utils.data.get_worker_info()
    if info is not None and info.dataset.transform is not None:
        info.dataset.transform.set_random_seed(seed)  # Albumentations tiene su propio RNG


def get_rng_state() -> dict:
    state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if "cuda" in state and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def save_torch_atomic(obj: object, path: Path) -> None:
    atomic_write(path, lambda tmp: torch.save(obj, tmp))


def load_checkpoint(out_dir: Path, map_location: str = "cpu") -> Optional[dict]:
    """Ultimo checkpoint legible. Si el ultimo esta dañado, prueba con el anterior."""
    for name in (LAST_CKPT, PREV_CKPT):
        path = out_dir / name
        if not path.exists():
            continue
        try:
            # weights_only=False: el checkpoint es nuestro y trae estado de RNG de numpy.
            ckpt = torch.load(path, map_location=map_location, weights_only=False)
            print(f"Checkpoint encontrado: {path} (epocas completas: {ckpt['epoch']})", flush=True)
            return ckpt
        except Exception as exc:
            print(f"AVISO: no se pudo leer {path} ({exc}); pruebo con el anterior.", flush=True)
    return None


def save_checkpoint(ckpt: dict, out_dir: Path) -> None:
    """Guarda atomicamente. Antes conserva el ultimo bueno como respaldo."""
    last = out_dir / LAST_CKPT
    if last.exists():
        # Renombrar, no copiar: el archivo anterior nunca queda a medio escribir.
        last.replace(out_dir / PREV_CKPT)
    save_torch_atomic(ckpt, last)


def set_encoder_trainable(model: torch.nn.Module, trainable: bool) -> None:
    for param in model.encoder.parameters():
        param.requires_grad = trainable


def build_optimizer(model: torch.nn.Module, lr: float, encoder_lr: float, weight_decay: float) -> torch.optim.Optimizer:
    groups = [
        {"params": list(model.encoder.parameters()), "lr": encoder_lr},
        {"params": list(model.decoder.parameters()) + list(model.segmentation_head.parameters()), "lr": lr},
    ]
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


class CombinedLoss(torch.nn.Module):
    """Entropia cruzada ponderada + Dice multiclase, ambas ignorando IGNORE_INDEX."""

    def __init__(self, class_weights: torch.Tensor):
        super().__init__()
        self.ce = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)
        self.dice = smp.losses.DiceLoss(mode="multiclass", from_logits=True, ignore_index=IGNORE_INDEX)

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.ce(logits, target) + self.dice(logits, target)


def _confusion(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    valid = target != IGNORE_INDEX
    index = NUM_CLASSES * target[valid] + pred[valid]
    return torch.bincount(index, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def make_loaders(args, train_rows: list[dict], val_rows: list[dict]) -> tuple[DataLoader, DataLoader]:
    train_ds = SegDataset(args.data_root, train_rows, TRAINING_SPLITS, build_train_transform())
    val_ds = SegDataset(args.data_root, val_rows, TRAINING_SPLITS)
    weights = [BOWFIRE_SAMPLE_FACTOR if r["source"] == SOURCE_BOWFIRE else 1.0 for r in train_rows]
    sampler = WeightedRandomSampler(weights, num_samples=len(train_rows), replacement=True)
    # persistent_workers=False: los trabajadores se reinician cada epoca con una semilla que
    # sale del RNG principal, asi la reanudacion reproduce los mismos aumentos.
    common = dict(
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=_worker_init,
        persistent_workers=False,
    )
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, sampler=sampler, drop_last=len(train_rows) > args.batch_size, **common)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, **common)
    return train_loader, val_loader


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, use_amp=False, freeze_encoder=False) -> dict:
    is_train = optimizer is not None
    model.train(is_train)
    if is_train and freeze_encoder:
        model.encoder.eval()  # no actualizar BatchNorm mientras el encoder esta congelado
    total_loss, total_batches = 0.0, 0
    matrix = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=device)
    amp_device = "cuda" if device.type == "cuda" else "cpu"
    with torch.set_grad_enabled(is_train):
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(amp_device, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, masks)
            if is_train:
                if not torch.isfinite(loss):
                    raise FloatingPointError("La perdida no es finita; revisa los datos o baja el learning rate")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            total_batches += 1
            matrix += _confusion(logits.detach().argmax(dim=1), masks)
    result = metrics_from_confusion(matrix.cpu().numpy())
    result["loss"] = total_loss / max(total_batches, 1)
    return result


def write_best_model(model, out_dir: Path, epoch: int, miou: float, manifest_hash: str) -> None:
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    payload = {
        "state_dict": state,
        "encoder_name": ENCODER_NAME,
        "classes": CLASSES,
        "input_size": [320, 320],
        "epoch": epoch,
        "val_miou": miou,
        "dataset_manifest_sha256": manifest_hash,
    }
    save_torch_atomic(payload, out_dir / BEST_MODEL)


def parse_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, default=Path("data/processed"))
    parser.add_argument("--manifest", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--out-dir", type=Path, default=Path("data/runs/default"), help="En Colab, una carpeta de Drive")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--encoder-lr", type=float, default=1e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--freeze-encoder-epochs", type=int, default=2)
    parser.add_argument("--patience", type=int, default=8)
    parser.add_argument("--min-delta", type=float, default=0.001)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    parser.add_argument("--limit-train", type=int, default=None)
    parser.add_argument("--limit-val", type=int, default=None)
    parser.add_argument("--no-pretrained", action="store_true", help="Solo para pruebas")
    parser.add_argument("--stop-after-epochs", type=int, default=None, help="Solo para pruebas: para limpiamente tras guardar esa epoca")
    return parser.parse_args(argv)


def train(args: argparse.Namespace) -> dict:
    args.out_dir = Path(args.out_dir)
    args.out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if args.device == "cuda" or (args.device == "auto" and torch.cuda.is_available()) else "cpu")
    use_amp = device.type == "cuda"
    torch.backends.cudnn.benchmark = use_amp

    manifest = read_manifest(args.manifest)
    manifest_hash = manifest_sha256(args.manifest)
    train_rows = limit_rows(rows_for_split(manifest, "train", TRAINING_SPLITS), args.limit_train)
    val_rows = limit_rows(rows_for_split(manifest, "val", TRAINING_SPLITS), args.limit_val)
    print(f"Particiones: train={len(train_rows)} val={len(val_rows)}. La de prueba no se abre aqui.", flush=True)

    seed_everything(args.seed)
    model = build_model(pretrained=not args.no_pretrained).to(device)
    optimizer = build_optimizer(model, args.lr, args.encoder_lr, args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(args.epochs, 1), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    weights_path = args.out_dir / CLASS_WEIGHTS
    if weights_path.exists():
        weights_info = json.loads(weights_path.read_text(encoding="utf-8"))
    else:
        weights_info = compute_class_weights(args.data_root, train_rows)
        atomic_write_json(weights_path, weights_info)
    print("Pesos por clase (de los pixeles de train):", dict(zip(CLASSES, np.round(weights_info["weights"], 3))), flush=True)
    criterion = CombinedLoss(torch.tensor(weights_info["weights"], dtype=torch.float32, device=device))

    state = {"epoch": 0, "best_miou": -1.0, "best_epoch": 0, "bad_epochs": 0, "history": [], "finished": False}
    ckpt = load_checkpoint(args.out_dir)
    if ckpt is not None:
        if ckpt["dataset_manifest_sha256"] != manifest_hash:
            raise ManifestMismatchError("El checkpoint se entreno con otro manifiesto; no se puede reanudar con este.")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        state = {k: copy.deepcopy(ckpt[k]) for k in state}
        set_rng_state(ckpt["rng"])
        if not state["finished"]:
            print(f"REANUDANDO: continuo en la epoca {state['epoch'] + 1} de {args.epochs}; mejor mIoU de val: {state['best_miou']:.4f}; "
                  f"epocas sin mejora: {state['bad_epochs']}", flush=True)
    else:
        print("Sin checkpoint previo: empiezo desde la epoca 1.", flush=True)

    if state["finished"]:
        print("El entrenamiento ya habia terminado. No hay nada que hacer.", flush=True)
        return state

    train_loader, val_loader = make_loaders(args, train_rows, val_rows)

    while state["epoch"] < args.epochs:
        epoch = state["epoch"] + 1
        start = time.time()
        freeze = epoch <= args.freeze_encoder_epochs
        set_encoder_trainable(model, not freeze)
        # Albumentations tiene su propio RNG: se siembra desde el principal, que se restaura al reanudar.
        train_loader.dataset.transform.set_random_seed(int(torch.randint(0, 2**31 - 1, (1,)).item()))
        train_m = run_epoch(model, train_loader, criterion, device, optimizer, scaler, use_amp, freeze)
        train_seconds = time.time() - start
        val_m = run_epoch(model, val_loader, criterion, device)
        scheduler.step()
        seconds = time.time() - start

        state["epoch"] = epoch
        improved = val_m["miou"] > state["best_miou"] + args.min_delta
        if improved:
            state.update(best_miou=val_m["miou"], best_epoch=epoch, bad_epochs=0)
            write_best_model(model, args.out_dir, epoch, val_m["miou"], manifest_hash)
        else:
            state["bad_epochs"] += 1
        state["history"].append(
            {
                "epoch": epoch,
                "train_loss": train_m["loss"],
                "val_loss": val_m["loss"],
                "train_miou": train_m["miou"],
                "val_miou": val_m["miou"],
                "val_pixel_accuracy": val_m["pixel_accuracy"],
                "val_iou_per_class": {k: v["iou"] for k, v in val_m["per_class"].items()},
                "lr_encoder": optimizer.param_groups[0]["lr"],
                "lr_decoder": optimizer.param_groups[1]["lr"],
                "seconds": seconds,
                "train_seconds": train_seconds,
            }
        )
        if state["bad_epochs"] >= args.patience:
            state["finished"] = True
            print(f"Early stopping: {args.patience} epocas sin mejorar el mIoU de validacion.", flush=True)
        if epoch >= args.epochs:
            state["finished"] = True

        ckpt = {
            **copy.deepcopy(state),
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "rng": get_rng_state(),
            "dataset_manifest_sha256": manifest_hash,
            "args": {k: str(v) for k, v in vars(args).items()},
        }
        save_checkpoint(ckpt, args.out_dir)
        atomic_write_json(args.out_dir / HISTORY, state["history"])

        recent = [h["seconds"] for h in state["history"][-5:]]
        remaining = (args.epochs - epoch) * float(np.mean(recent))
        print(
            f"Epoca {epoch}/{args.epochs} | train loss {train_m['loss']:.4f} mIoU {train_m['miou']:.4f} | "
            f"val loss {val_m['loss']:.4f} mIoU {val_m['miou']:.4f} | mejor {state['best_miou']:.4f} (epoca {state['best_epoch']}) | "
            f"sin mejora {state['bad_epochs']}/{args.patience} | tiempo medido {seconds:.1f} s | "
            f"restante estimado si no hay early stopping: {remaining / 60:.1f} min | checkpoint guardado",
            flush=True,
        )
        if state["finished"]:
            break
        if args.stop_after_epochs is not None and epoch >= args.stop_after_epochs:
            print(f"Parada solicitada tras la epoca {epoch} (solo para pruebas).", flush=True)
            return state

    print(f"Fin. Mejor mIoU de validacion: {state['best_miou']:.4f} en la epoca {state['best_epoch']}.", flush=True)
    return state


def main() -> None:
    train(parse_args())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/export.py
"""Exporta el mejor checkpoint a .pt y .onnx, y escribe manifest.json y hashes.sha256.

Uso:
    python ml/export.py --checkpoint <carpeta>/kinetia_seg.pt --midas models/midas_small.onnx --out-dir models
"""
from __future__ import annotations

import argparse
import datetime as dt
import inspect
import shutil
import sys
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch

from dataset import (
    CLASSES,
    DEFAULT_MANIFEST,
    INPUT_SIZE,
    MEAN,
    NUM_CLASSES,
    STD,
    atomic_write,
    atomic_write_json,
    file_sha256,
    manifest_sha256,
)
from model import build_model

OPSET = 17
VERIFY_TOLERANCE = 1e-4
SEG_ONNX = "kinetia_seg.onnx"
SEG_PT = "kinetia_seg.pt"
MIDAS_ONNX = "midas_small.onnx"


def load_trained(path: Path) -> tuple[torch.nn.Module, dict]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(pretrained=False)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload


def export_onnx(model: torch.nn.Module, path: Path) -> None:
    """Grafo con N dinamico, entradas `input` y salidas `logits`, sin softmax."""
    dummy = torch.zeros(2, 3, INPUT_SIZE, INPUT_SIZE)
    kwargs = dict(
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "N"}, "logits": {0: "N"}},
        opset_version=OPSET,
        do_constant_folding=True,
    )
    # PyTorch >= 2.9 usa por defecto otro exportador; se pide el clasico cuando existe la opcion,
    # y no se reinstala PyTorch en Colab.
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        kwargs["dynamo"] = False
    atomic_write(path, lambda tmp: torch.onnx.export(model, dummy, tmp, **kwargs))


def check_onnx_signature(path: Path) -> None:
    graph = onnx.load(str(path))
    onnx.checker.check_model(graph)
    opset = max(o.version for o in graph.opset_import if o.domain in ("", "ai.onnx"))
    if opset < OPSET:
        raise RuntimeError(f"Opset {opset} menor que {OPSET}")
    names_in = [i.name for i in graph.graph.input]
    names_out = [o.name for o in graph.graph.output]
    if names_in != ["input"] or names_out != ["logits"]:
        raise RuntimeError(f"Nombres incorrectos: {names_in} -> {names_out}")
    if any(n.op_type == "Softmax" for n in graph.graph.node):
        raise RuntimeError("El ONNX contiene un Softmax: el contrato pide logits")


def verify_pt_vs_onnx(model: torch.nn.Module, path: Path, tolerance: float = VERIFY_TOLERANCE) -> float:
    """Compara con lotes de distinto tamano. Devuelve la mayor diferencia absoluta."""
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    rng = np.random.default_rng(0)
    worst = 0.0
    for n in (1, 2, 3):
        x = rng.normal(size=(n, 3, INPUT_SIZE, INPUT_SIZE)).astype(np.float32)
        with torch.no_grad():
            expected = model(torch.from_numpy(x)).numpy()
        got = session.run(["logits"], {"input": x})[0]
        if got.shape != (n, NUM_CLASSES, INPUT_SIZE, INPUT_SIZE) or got.dtype != np.float32:
            raise RuntimeError(f"Salida ONNX inesperada: {got.shape} {got.dtype}")
        difference = np.abs(expected - got)
        if not np.isfinite(difference).all():  # NaN pasaria un max() y parecería que coinciden
            raise RuntimeError("Hay valores no finitos en la salida del .pt o del .onnx")
        worst = max(worst, float(difference.max()))
    if worst > tolerance:
        raise RuntimeError(f"ONNX y .pt difieren en {worst:.2e} (tolerancia {tolerance:.0e})")
    return worst


def write_manifest_json(out_dir: Path, midas: Path, manifest_csv: Path, version: str) -> dict:
    seg = out_dir / SEG_ONNX
    data = {
        "version": version,
        "created_at": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "classes": CLASSES,
        "input_size": [INPUT_SIZE, INPUT_SIZE],
        "mean": [round(float(v), 3) for v in MEAN],
        "std": [round(float(v), 3) for v in STD],
        "files": {
            SEG_ONNX: {"sha256": file_sha256(seg), "bytes": seg.stat().st_size},
            MIDAS_ONNX: {"sha256": file_sha256(midas), "bytes": Path(midas).stat().st_size},
        },
        "dataset_manifest_sha256": manifest_sha256(manifest_csv),
    }
    atomic_write_json(out_dir / "manifest.json", data)
    return data


def write_hashes(out_dir: Path) -> None:
    lines = "".join(f"{file_sha256(out_dir / name)}  {name}\n" for name in (SEG_PT, SEG_ONNX))

    def writer(tmp: str) -> None:
        with open(tmp, "w", encoding="ascii", newline="\n") as f:
            f.write(lines)

    atomic_write(out_dir / "hashes.sha256", writer)


def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--midas", type=Path, required=True, help="midas_small.onnx (ver fetch_depth_model.py)")
    parser.add_argument("--manifest", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--out-dir", type=Path, default=Path("models"))
    parser.add_argument("--version", default="seg-1.0.0")
    args = parser.parse_args(argv)

    args.out_dir.mkdir(parents=True, exist_ok=True)
    model, payload = load_trained(args.checkpoint)
    if payload["dataset_manifest_sha256"] != manifest_sha256(args.manifest):
        sys.exit("ERROR: el checkpoint se entreno con otro manifiesto de datos.")

    pt_out = args.out_dir / SEG_PT
    if args.checkpoint.resolve() != pt_out.resolve():
        atomic_write(pt_out, lambda tmp: shutil.copyfile(args.checkpoint, tmp))
    onnx_out = args.out_dir / SEG_ONNX
    export_onnx(model, onnx_out)
    check_onnx_signature(onnx_out)
    worst = verify_pt_vs_onnx(model, onnx_out)
    print(f".pt vs .onnx: diferencia maxima {worst:.2e} (tolerancia {VERIFY_TOLERANCE:.0e}), N dinamico comprobado")

    manifest = write_manifest_json(args.out_dir, args.midas, args.manifest, args.version)
    write_hashes(args.out_dir)
    print(f"SHA-256 kinetia_seg.pt:   {file_sha256(pt_out)}")
    print(f"SHA-256 kinetia_seg.onnx: {manifest['files'][SEG_ONNX]['sha256']}")
    print(f"Escritos en {args.out_dir}: {SEG_PT}, {SEG_ONNX}, manifest.json, hashes.sha256")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/evaluate.py
"""Evalua el .onnx exportado sobre la particion de prueba y escribe metrics.json.

Uso:
    python ml/evaluate.py --models-dir models --data-root data/test                 # escribe metrics.json
    python ml/evaluate.py --models-dir models --data-root data/test --verify models/metrics.json
    python ml/evaluate.py --models-dir models --data-root data/test --limit 50      # NO oficial

Las metricas se calculan a 320x320 (la resolucion del modelo) y omiten IGNORE_INDEX.
"""
from __future__ import annotations

import argparse
import datetime as dt
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort

from dataset import (
    CLASSES,
    DEFAULT_MANIFEST,
    EVALUATION_SPLITS,
    FUEGO_LUZ,
    NUM_CLASSES,
    SOURCE_BOWFIRE,
    SOURCE_COCO,
    atomic_write_json,
    file_sha256,
    image_path,
    limit_rows,
    manifest_sha256,
    mask_path,
    preprocess_seg,
    read_image_bgr,
    read_manifest,
    read_mask,
    resize_mask,
    rows_for_split,
)
from seg_metrics import confusion_update, metrics_from_confusion

VERIFY_TOLERANCE = 1e-3
NOT_OFFICIAL_BANNER = (
    "\n" + "=" * 72 + "\n"
    "  AVISO: se uso --limit. Estas cifras NO son las oficiales: no sirven para el\n"
    "  documento tecnico ni para la sustentacion. Las oficiales salen sin --limit.\n"
    + "=" * 72 + "\n"
)


def evaluate(onnx_path: Path, rows: list[dict], data_root: Path, batch_size: int = 8) -> dict:
    """Evalua y devuelve metricas, tiempos y una matriz de confusion por fuente."""
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    by_source = {SOURCE_COCO: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64), SOURCE_BOWFIRE: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)}
    start = time.time()
    last_print = start
    for i in range(0, len(rows), batch_size):
        batch = rows[i : i + batch_size]
        x = np.concatenate([preprocess_seg(read_image_bgr(data_root, r)) for r in batch], axis=0)
        logits = session.run(["logits"], {"input": x})[0]
        pred = logits.argmax(axis=1)
        for row, p in zip(batch, pred):
            confusion_update(by_source[row["source"]], resize_mask(read_mask(data_root, row)), p)
        if time.time() - last_print > 30:
            last_print = time.time()
            done = i + len(batch)
            print(f"  {done}/{len(rows)} imagenes; {(last_print - start) / done:.3f} s/imagen", flush=True)
    return {"by_source": by_source, "seconds": time.time() - start}


def build_metrics(onnx_path: Path, manifest_hash: str, matrix: np.ndarray, num_images: int) -> dict:
    """Solo los campos del esquema de metrics.json del contrato."""
    m = metrics_from_confusion(matrix)
    return {
        "model_sha256": file_sha256(onnx_path),
        "dataset_manifest_sha256": manifest_hash,
        "split": "test",
        "num_images": num_images,
        "evaluated_at": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "miou": m["miou"],
        "dice_macro": m["dice_macro"],
        "pixel_accuracy": m["pixel_accuracy"],
        "per_class": m["per_class"],
        "confusion_matrix": matrix.tolist(),
        "classes": CLASSES,
    }


def compare_metrics(expected: dict, got: dict, tolerance: float = VERIFY_TOLERANCE) -> list[str]:
    """Lista de diferencias: hashes y conteo deben ser iguales; las cifras, dentro de tolerancia."""
    problems: list[str] = []
    for key in ("model_sha256", "dataset_manifest_sha256", "split", "num_images", "classes"):
        if expected.get(key) != got.get(key):
            problems.append(f"{key}: archivo={expected.get(key)} calculado={got.get(key)}")
    for key in ("miou", "dice_macro", "pixel_accuracy"):
        if not math.isclose(expected[key], got[key], abs_tol=tolerance):
            problems.append(f"{key}: archivo={expected[key]:.6f} calculado={got[key]:.6f}")
    for name in CLASSES:
        for key in ("iou", "dice"):
            a, b = expected["per_class"][name][key], got["per_class"][name][key]
            if not math.isclose(a, b, abs_tol=tolerance):
                problems.append(f"{name}.{key}: archivo={a:.6f} calculado={b:.6f}")
    # La matriz se compara en proporcion del total, porque los conteos son enteros grandes.
    a = np.asarray(expected["confusion_matrix"], dtype=np.float64)
    b = np.asarray(got["confusion_matrix"], dtype=np.float64)
    if a.shape != b.shape:
        problems.append(f"confusion_matrix: formas {a.shape} y {b.shape}")
    else:
        worst = float(np.abs(a / max(a.sum(), 1) - b / max(b.sum(), 1)).max())
        if worst > tolerance:
            problems.append(f"confusion_matrix: diferencia maxima de proporcion {worst:.6f}")
    return problems


def print_summary(metrics: dict, by_source: dict, seconds: float, limited: bool) -> None:
    print(f"\nImagenes: {metrics['num_images']} | tiempo medido: {seconds:.1f} s ({seconds / max(metrics['num_images'], 1):.3f} s/imagen)")
    print(f"mIoU {metrics['miou']:.4f} | Dice macro {metrics['dice_macro']:.4f} | exactitud de pixel {metrics['pixel_accuracy']:.4f}")
    for name in CLASSES:
        c = metrics["per_class"][name]
        print(f"  {name:<11} IoU {c['iou']:.4f}  Dice {c['dice']:.4f}  soporte {c['support_pixels']}")
    fire = by_source[SOURCE_BOWFIRE]
    if fire.sum():
        recall = fire[FUEGO_LUZ, FUEGO_LUZ] / fire[FUEGO_LUZ].sum()
        print(f"Solo imagenes de fuego (BoWFire): {recall:.4f} de los pixeles de fuego se predicen como fuego_luz.")
        print("  Es recall: en BoWFire el resto de la imagen se ignora, asi que no mide falsos positivos.")
    light = by_source[SOURCE_COCO]
    if light[FUEGO_LUZ].sum():
        print(f"Solo COCO-Stuff: {light[FUEGO_LUZ, FUEGO_LUZ] / light[FUEGO_LUZ].sum():.4f} de los pixeles de luz se predicen como fuego_luz.")
    if limited:
        print(NOT_OFFICIAL_BANNER)


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--models-dir", type=Path, default=Path("models"))
    parser.add_argument("--data-root", type=Path, default=Path("data/test"))
    parser.add_argument("--manifest", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--out", type=Path, default=None, help="Por defecto, <models-dir>/metrics.json")
    parser.add_argument("--verify", type=Path, default=None, help="Compara con este metrics.json, sin escribir")
    parser.add_argument("--limit", type=int, default=None, help="Solo N imagenes. Las cifras NO son las oficiales")
    parser.add_argument("--batch-size", type=int, default=8)
    args = parser.parse_args(argv)

    onnx_path = args.models_dir / "kinetia_seg.onnx"
    model_manifest_path = args.models_dir / "manifest.json"
    for path in (onnx_path, model_manifest_path, args.manifest):
        if not path.exists():
            print(f"ERROR: no existe {path}", file=sys.stderr)
            return 2
    model_manifest = json.loads(model_manifest_path.read_text(encoding="utf-8"))
    manifest_hash = manifest_sha256(args.manifest)
    if model_manifest["dataset_manifest_sha256"] != manifest_hash:
        print(
            "ERROR: el hash de ml/splits/manifest.csv no coincide con el de manifest.json.\n"
            f"  manifest.csv : {manifest_hash}\n  manifest.json: {model_manifest['dataset_manifest_sha256']}",
            file=sys.stderr,
        )
        return 3
    actual = file_sha256(onnx_path)
    if model_manifest["files"]["kinetia_seg.onnx"]["sha256"] != actual:
        print(f"ERROR: {onnx_path} no coincide con el hash de manifest.json ({actual})", file=sys.stderr)
        return 3

    rows = rows_for_split(read_manifest(args.manifest), EVALUATION_SPLITS[0], EVALUATION_SPLITS)
    limited = args.limit is not None
    if limited:
        print(NOT_OFFICIAL_BANNER, flush=True)
        rows = limit_rows(rows, args.limit)

    missing = [r["image_id"] for r in rows if not (image_path(args.data_root, r).exists() and mask_path(args.data_root, r).exists())]
    if missing:
        print(f"ERROR: faltan {len(missing)} imagenes en {args.data_root} (ej. {missing[:3]}). Ejecuta ml/fetch_test_set.py.", file=sys.stderr)
        return 2

    result = evaluate(onnx_path, rows, args.data_root, args.batch_size)
    matrix = sum(result["by_source"].values())
    metrics = build_metrics(onnx_path, manifest_hash, matrix, len(rows))
    print_summary(metrics, result["by_source"], result["seconds"], limited)

    if args.verify is not None:
        expected = json.loads(args.verify.read_text(encoding="utf-8"))
        problems = compare_metrics(expected, metrics)
        if problems:
            print(f"VERIFICACION FALLIDA contra {args.verify}:")
            for problem in problems:
                print(f"  - {problem}")
            return 1
        print(f"VERIFICACION CORRECTA: coincide con {args.verify} (tolerancia {VERIFY_TOLERANCE}).")
        return 0

    out = args.out or args.models_dir / "metrics.json"
    if limited and out.resolve().parent == Path("models").resolve():
        print("ERROR: unas cifras con --limit no pueden escribirse en models/.", file=sys.stderr)
        return 2
    atomic_write_json(out, metrics)
    print(f"Escrito {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ml/fetch_depth_model.py
"""Descarga MiDaS v2.1 small ONNX de la release oficial y verifica su SHA-256.

No es un modelo entrenado por el equipo: se usa preentrenado, solo para el parallax.

Uso:
    python ml/fetch_depth_model.py                # escribe models/midas_small.onnx
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import onnxruntime as ort

from dataset import file_sha256
from prepare_data import download

MIDAS_URL = "https://github.com/isl-org/MiDaS/releases/download/v2_1/model-small.onnx"
MIDAS_BYTES = 66764249
# Calculado sobre el archivo descargado de la release oficial el 2026-11-02.
# La release no publica hash; este fija el archivo para que todas las maquinas usen el mismo.
MIDAS_SHA256 = "2d8c6cb8f415229daf1eb041024208e2608c9f98e17c81cc7c6ecb449c56fd58"


def fetch(out: Path) -> Path:
    download(MIDAS_URL, out, MIDAS_BYTES)
    actual = file_sha256(out)
    if actual != MIDAS_SHA256:
        out.unlink()
        raise RuntimeError(f"SHA-256 inesperado en MiDaS: {actual}; se borro el archivo.")
    session = ort.InferenceSession(str(out), providers=["CPUExecutionProvider"])
    shape = session.get_inputs()[0].shape
    if list(shape[1:]) != [3, 256, 256]:
        raise RuntimeError(f"Entrada de MiDaS inesperada: {shape}")
    return out


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=Path("models/midas_small.onnx"))
    args = parser.parse_args(argv)
    fetch(args.out)
    print(f"{args.out}: {args.out.stat().st_size} bytes, SHA-256 {MIDAS_SHA256}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ml/fetch_test_set.py
"""Descarga SOLO el conjunto de prueba a data/test, para correr evaluate.py en este portatil.

Baja cada imagen de prueba por separado, y una vez el zip de mascaras de COCO-Stuff (659 MB)
y el de BoWFire (291 MB), porque esos autores no ofrecen mascaras sueltas. Solo guarda lo de
prueba. Con --clean-raw borra los zip al terminar.

Uso:
    python ml/fetch_test_set.py
    python ml/evaluate.py --models-dir models --data-root data/test --verify models/metrics.json
"""
from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path

from dataset import DEFAULT_MANIFEST, EVALUATION_SPLITS, image_path, limit_rows, mask_path, read_manifest, rows_for_split
from prepare_data import check_prepared, prepare_rows


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=Path("data/test"))
    parser.add_argument("--raw-dir", type=Path, default=Path("data/raw"))
    parser.add_argument("--manifest", type=Path, default=DEFAULT_MANIFEST)
    parser.add_argument("--threads", type=int, default=16)
    parser.add_argument("--limit", type=int, default=None, help="Solo N imagenes: sirve para probar, no para las cifras oficiales")
    parser.add_argument("--clean-raw", action="store_true", help="Borra los zip descargados al terminar")
    args = parser.parse_args(argv)

    rows = rows_for_split(read_manifest(args.manifest), EVALUATION_SPLITS[0], EVALUATION_SPLITS)
    rows = limit_rows(rows, args.limit)
    prepare_rows(rows, args.out, args.raw_dir, args.threads)
    missing = check_prepared(rows, args.out)
    if missing:
        print(f"ERROR: faltan {len(missing)} imagenes: {missing[:5]}", file=sys.stderr)
        return 1
    size = sum(image_path(args.out, r).stat().st_size + mask_path(args.out, r).stat().st_size for r in rows)
    print(f"Conjunto de prueba listo en {args.out}: {len(rows)} imagenes, {size / 1e6:.0f} MB (medido)")
    if args.clean_raw:
        shutil.rmtree(args.raw_dir, ignore_errors=True)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ml/validate_artifacts.py
"""Comprueba que los 5 archivos que entrega el notebook cumplen los esquemas del contrato."""
from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

from dataset import CLASSES, file_sha256

HEX64 = re.compile(r"^[0-9a-f]{64}$")
ISO = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")
ARTIFACTS = ("kinetia_seg.pt", "kinetia_seg.onnx", "metrics.json", "manifest.json", "hashes.sha256")


def validate_manifest(data: dict) -> list[str]:
    p: list[str] = []
    keys = {"version", "created_at", "classes", "input_size", "mean", "std", "files", "dataset_manifest_sha256"}
    if set(data) != keys:
        p.append(f"manifest.json: campos {sorted(set(data) ^ keys)} sobran o faltan")
        return p
    if data["classes"] != CLASSES:
        p.append("manifest.json: clases incorrectas")
    if data["input_size"] != [320, 320]:
        p.append("manifest.json: input_size incorrecto")
    if data["mean"] != [0.485, 0.456, 0.406] or data["std"] != [0.229, 0.224, 0.225]:
        p.append("manifest.json: mean/std incorrectos")
    if not ISO.match(data["created_at"]):
        p.append("manifest.json: created_at no es ISO 8601 UTC")
    if set(data["files"]) != {"kinetia_seg.onnx", "midas_small.onnx"}:
        p.append("manifest.json: files incorrecto")
    for name, info in data["files"].items():
        if not HEX64.match(info["sha256"]) or not isinstance(info["bytes"], int) or info["bytes"] <= 0:
            p.append(f"manifest.json: entrada invalida para {name}")
    if not HEX64.match(data["dataset_manifest_sha256"]):
        p.append("manifest.json: dataset_manifest_sha256 invalido")
    return p


def validate_metrics(data: dict) -> list[str]:
    p: list[str] = []
    keys = {
        "model_sha256", "dataset_manifest_sha256", "split", "num_images", "evaluated_at", "miou",
        "dice_macro", "pixel_accuracy", "per_class", "confusion_matrix", "classes",
    }
    if set(data) != keys:
        p.append(f"metrics.json: campos {sorted(set(data) ^ keys)} sobran o faltan")
        return p
    if not HEX64.match(data["model_sha256"]) or not HEX64.match(data["dataset_manifest_sha256"]):
        p.append("metrics.json: hash invalido")
    if data["split"] != "test":
        p.append("metrics.json: split debe ser test")
    if data["classes"] != CLASSES:
        p.append("metrics.json: clases incorrectas")
    if not ISO.match(data["evaluated_at"]):
        p.append("metrics.json: evaluated_at no es ISO 8601 UTC")
    if not isinstance(data["num_images"], int) or data["num_images"] <= 0:
        p.append("metrics.json: num_images invalido")
    for key in ("miou", "dice_macro", "pixel_accuracy"):
        if not 0.0 <= data[key] <= 1.0:
            p.append(f"metrics.json: {key} fuera de [0, 1]")
    if list(data["per_class"]) != CLASSES:
        p.append("metrics.json: per_class debe traer las 7 clases en orden")
    else:
        for name, c in data["per_class"].items():
            if set(c) != {"iou", "dice", "support_pixels"}:
                p.append(f"metrics.json: campos invalidos en {name}")
            elif not (0.0 <= c["iou"] <= 1.0 and 0.0 <= c["dice"] <= 1.0 and isinstance(c["support_pixels"], int)):
                p.append(f"metrics.json: valores invalidos en {name}")
    matrix = data["confusion_matrix"]
    if len(matrix) != 7 or any(len(r) != 7 or not all(isinstance(v, int) for v in r) for r in matrix):
        p.append("metrics.json: confusion_matrix debe ser 7x7 de enteros")
    elif [sum(r) for r in matrix] != [data["per_class"][n]["support_pixels"] for n in CLASSES]:
        p.append("metrics.json: el soporte no coincide con la matriz")
    return p


def validate_dir(directory: Path) -> list[str]:
    directory = Path(directory)
    problems = [f"Falta {name}" for name in ARTIFACTS if not (directory / name).exists()]
    if problems:
        return problems
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
    metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
    problems += validate_manifest(manifest) + validate_metrics(metrics)
    onnx_hash = file_sha256(directory / "kinetia_seg.onnx")
    if manifest["files"]["kinetia_seg.onnx"]["sha256"] != onnx_hash:
        problems.append("manifest.json no coincide con kinetia_seg.onnx")
    if metrics["model_sha256"] != onnx_hash:
        problems.append("metrics.json no se calculo sobre este kinetia_seg.onnx")
    if metrics["dataset_manifest_sha256"] != manifest["dataset_manifest_sha256"]:
        problems.append("metrics.json y manifest.json usan manifiestos de datos distintos")
    expected_hashes = "".join(f"{file_sha256(directory / n)}  {n}\n" for n in ("kinetia_seg.pt", "kinetia_seg.onnx"))
    if (directory / "hashes.sha256").read_bytes().decode("ascii") != expected_hashes:
        problems.append("hashes.sha256 no coincide con los modelos")
    return problems


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("directory", type=Path)
    args = parser.parse_args()
    problems = validate_dir(args.directory)
    for problem in problems:
        print(f"ERROR: {problem}")
    if not problems:
        print(f"Los 5 archivos de {args.directory} cumplen los esquemas del contrato.")
    return 1 if problems else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
# Particion fija (ml/splits/manifest.csv), comprimida para no ocupar 15 000 lineas.
# Es la misma del repositorio: https://github.com/erikabonilla1912/kinetia
import base64, gzip, hashlib
MANIFEST_SHA256 = "fb8a7cc42dac1b726111abffe315e974014eb3880f160d9b0ef3095d3ba47b9d"
_MANIFEST_B64 = """
...
"""
_data = gzip.decompress(base64.b64decode(_MANIFEST_B64))
assert hashlib.sha256(_data).hexdigest() == MANIFEST_SHA256, "Manifiesto incrustado danado"
Path("ml/splits").mkdir(parents=True, exist_ok=True)
Path("ml/splits/manifest.csv").write_bytes(_data)
print("Manifiesto escrito; SHA-256:", MANIFEST_SHA256, "| filas:", _data.count(b"\n") - 1)


## 3. Dependencias y GPU

In [ ]:
# PyTorch ya viene en Colab y NO se reinstala. Solo se instala lo que falta o difiere.
# Colab trae varias variantes de OpenCV que comparten la carpeta cv2, y por eso se quitan antes;
# si no, se mezclarían versiones y el resize dejaría de ser el del contrato.
from importlib import metadata

pins = []
for line in Path("ml/requirements.txt").read_text(encoding="utf-8").splitlines():
    spec = line.split("#")[0].strip()
    if not spec:
        continue
    name = spec.split("==")[0]
    if name in ("torch", "torchvision", "numpy", "pytest", "nbconvert", "nbformat", "ipykernel", "jupyter-client"):
        continue
    if IN_COLAB and "# colab" not in line:
        continue
    pins.append(spec)


def installed(spec):
    name, version = spec.split("==")
    try:
        return metadata.version(name) == version
    except metadata.PackageNotFoundError:
        return False


missing = [p for p in pins if not installed(p)]
if missing:
    print("Instalando:", missing)
    if IN_COLAB and any(p.startswith("opencv") for p in missing):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python", "opencv-contrib-python", "opencv-python-headless"], check=False)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencias ya instaladas.")

# Se comprueba en un proceso nuevo: es lo que verán los scripts.
probe = subprocess.run(
    [sys.executable, "-c", "import numpy,cv2,onnx,onnxruntime,albumentations,segmentation_models_pytorch as s,torch;print(numpy.__version__,cv2.__version__,onnx.__version__,onnxruntime.__version__,albumentations.__version__,s.__version__,torch.__version__)"],
    capture_output=True, text=True, env={**os.environ, "NO_ALBUMENTATIONS_UPDATE": "1"},
)
if probe.returncode != 0:
    raise RuntimeError("Las dependencias no se importan:\n" + probe.stderr[-2000:])
versions = probe.stdout.split()
print("numpy, cv2, onnx, onnxruntime, albumentations, smp, torch =", versions)
if versions[1] != "4.10.0":
    raise RuntimeError(f"OpenCV debe ser 4.10.0 y es {versions[1]}. Ve a Entorno de ejecución > Reiniciar sesión y pulsa Ejecutar todo otra vez.")
# Si pip cambió numpy con este kernel ya usándolo, hay que reiniciar; al repetir ya no se instala nada.
if "numpy" in sys.modules and sys.modules["numpy"].__version__ != versions[0]:
    raise RuntimeError("La instalación cambió numpy. Ve a Entorno de ejecución > Reiniciar sesión y pulsa Ejecutar todo otra vez: no se pierde nada.")

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
elif SMOKE_TEST:
    print("SMOKE_TEST: se usa CPU.")
else:
    raise RuntimeError(
        "NO HAY GPU, y entrenar sin ella tardaría días. Para activarla: menú Entorno de ejecución > "
        "Cambiar tipo de entorno de ejecución > Acelerador de hardware: GPU T4 > Guardar. "
        "Después pulsa Entorno de ejecución > Ejecutar todo. Si Colab dice que no hay GPU disponible, "
        "se agotó la cuota gratuita: inténtalo más tarde (ver docs/COLAB_PASO_A_PASO.md)."
    )

## 4. Google Drive

In [ ]:
# Drive guarda lo que debe sobrevivir a un corte: el dataset en un tar y los checkpoints.
if SMOKE_TEST:
    DRIVE_DIR = WORK / "drive_simulado"
    print("SMOKE_TEST: sin Drive; se usa una carpeta local que hace sus veces.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/kinetia")
RUN_DIR = DRIVE_DIR / "run"
TAR_PATH = DRIVE_DIR / "dataset" / "kinetia_dataset_320.tar"
TAR_META = TAR_PATH.with_suffix(".json")
RUN_DIR.mkdir(parents=True, exist_ok=True)
TAR_PATH.parent.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(DRIVE_DIR)
print(f"Carpeta persistente: {DRIVE_DIR} | libre: {usage.free / 1e9:.1f} GB de {usage.total / 1e9:.1f} GB")

## 5. Datos

- **COCO-Stuff 164k:** unas 15 000 imágenes elegidas por script para reforzar las clases escasas. Las particiones oficiales de COCO se mezclan y se reparten de nuevo con semilla fija (`seed=42`), porque `test2017` no tiene máscaras públicas y porque así BoWFire se reparte en las tres.
- **BoWFire** (Chino et al., SIBGRAPI 2015): 119 imágenes de fuego con máscara manual. En ellas **solo cuenta el fuego**; el resto es `IGNORE_INDEX=255`. Las fotos casi idénticas se mantienen en la misma partición para que no haya fuga entre entrenamiento y prueba.
- **Licencias:** COCO usa imágenes de Flickr con licencias variadas; BoWFire declara imágenes de Flickr bajo Creative Commons y pide citar su artículo.

La primera sesión descarga y preprocesa, y guarda un tar en Drive. Las siguientes solo lo extraen a `/content`, porque leer de Drive directamente es lento.

In [ ]:
from dataset import limit_rows, manifest_sha256, read_manifest
from prepare_data import check_prepared, extract_tar, make_tar, prepare_rows

manifest = read_manifest("ml/splits/manifest.csv")
if SMOKE_TEST:
    rows = [r for split, n in SMOKE_LIMITS.items() for r in limit_rows([m for m in manifest if m["split"] == split], n)]
else:
    rows = manifest
print(f"Imágenes a usar en esta ejecución: {len(rows)}")
t0 = time.time()


def tar_is_valid():
    if not (TAR_PATH.exists() and TAR_META.exists()):
        return False
    meta = json.loads(TAR_META.read_text(encoding="utf-8"))
    return meta["manifest_sha256"] == MANIFEST_SHA256 and meta["bytes"] == TAR_PATH.stat().st_size and meta["num_images"] == len(rows)


if not check_prepared(rows, DATA_DIR):
    print("Datos ya presentes en esta sesión.")
elif tar_is_valid():
    print(f"Extrayendo {TAR_PATH} ...", flush=True)
    extract_tar(TAR_PATH, DATA_DIR)
    print(f"Extraído en {time.time() - t0:.0f} s (medido).")
else:
    print("Primera vez: descargo y preproceso. El tiempo depende de la red; se imprime el medido.", flush=True)
    prepare_rows(rows, DATA_DIR, RAW_DIR)
    local_tar = WORK / TAR_PATH.name
    info = make_tar(DATA_DIR, rows, local_tar)
    info["manifest_sha256"] = MANIFEST_SHA256
    free = shutil.disk_usage(TAR_PATH.parent).free
    print(f"Tar: {info['bytes'] / 1e9:.2f} GB (medido); libre en la carpeta persistente: {free / 1e9:.1f} GB")
    if free < info["bytes"] * 1.1:
        raise RuntimeError(f"No cabe el tar en Drive: necesita {info['bytes'] / 1e9:.2f} GB. Libera espacio y vacía la papelera.")
    tmp = TAR_PATH.with_name(TAR_PATH.name + ".tmp")
    shutil.copyfile(local_tar, tmp)
    os.replace(tmp, TAR_PATH)
    TAR_META.write_text(json.dumps(info, indent=2), encoding="utf-8")
    local_tar.unlink()
    print(f"Tar guardado en {TAR_PATH}. Total: {(time.time() - t0) / 60:.1f} min (medido).")

missing = check_prepared(rows, DATA_DIR)
if missing:
    raise RuntimeError(f"Faltan {len(missing)} imágenes tras preparar: {missing[:5]}")
print("Datos listos.")

## 6. Análisis exploratorio

In [ ]:
# Análisis exploratorio sobre las máscaras reales. Solo se abre train: la prueba no se mira.
import collections
import numpy as np
import matplotlib.pyplot as plt
from dataset import CLASSES, IGNORE_INDEX, NUM_CLASSES, SOURCE_BOWFIRE, image_path, read_image_bgr, read_mask

COLORS = ["#3A3F4B", "#4FC3F7", "#2979FF", "#66BB6A", "#FF7043", "#EC407A", "#FFCA28"]  # contrato, sección 1
table = collections.Counter((r["split"], r["source"]) for r in manifest)
print("Imágenes por partición y fuente (manifiesto completo):")
for key in sorted(table):
    print(f"  {key[0]:<5} {key[1]:<11} {table[key]}")

train_rows = [r for r in rows if r["split"] == "train"]
pixels = np.zeros(256, dtype=np.int64)
presence = np.zeros(NUM_CLASSES, dtype=np.int64)
for r in train_rows:
    h = np.bincount(read_mask(DATA_DIR, r).ravel(), minlength=256)
    pixels += h
    presence += (h[:NUM_CLASSES] > 0).astype(np.int64)
labeled = pixels[:NUM_CLASSES].sum()
print(f"\nIgnorados (255): {pixels[IGNORE_INDEX] / pixels.sum():.2%} de los píxeles de train")
for i, name in enumerate(CLASSES):
    print(f"  {name:<11} {pixels[i] / labeled:7.2%} de los píxeles etiquetados | presente en {presence[i]} imágenes")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].barh(CLASSES[::-1], (pixels[:NUM_CLASSES] / labeled * 100)[::-1], color=COLORS[::-1])
axes[0].set_xscale("log"); axes[0].set_xlabel("% de píxeles etiquetados (escala log)"); axes[0].set_title("Píxeles por clase en train")
axes[1].barh(CLASSES[::-1], presence[::-1], color=COLORS[::-1])
axes[1].set_xlabel("imágenes"); axes[1].set_title("Imágenes de train con la clase")
plt.tight_layout(); plt.show()

palette = np.array([tuple(int(c[i:i + 2], 16) for i in (1, 3, 5)) for c in COLORS], dtype=np.uint8)
samples = [r for r in train_rows if r["source"] == SOURCE_BOWFIRE][:2] + [r for r in train_rows if r["source"] != SOURCE_BOWFIRE][:4]
fig, axes = plt.subplots(2, len(samples), figsize=(2.6 * len(samples), 5.2))
for col, r in enumerate(samples):
    mask = read_mask(DATA_DIR, r)
    shown = palette[np.where(mask == IGNORE_INDEX, 0, mask)].copy()
    shown[mask == IGNORE_INDEX] = 255  # blanco = ignorado
    axes[0, col].imshow(read_image_bgr(DATA_DIR, r)[..., ::-1]); axes[0, col].set_title(r["source"], fontsize=8)
    axes[1, col].imshow(shown)
    axes[0, col].axis("off"); axes[1, col].axis("off")
plt.suptitle("Muestras de train: en blanco lo ignorado (255)"); plt.tight_layout(); plt.show()

## 7. Preprocesamiento y aumentos

In [ ]:
# La entrada es exactamente la del contrato: RGB, resize directo INTER_LINEAR a 320×320,
# /255 y normalización de ImageNet. Los aumentos solo se aplican en entrenamiento.
from dataset import SegDataset, preprocess_seg
from train import build_train_transform

r = train_rows[0]
x = preprocess_seg(read_image_bgr(DATA_DIR, r))
print("Forma:", x.shape, "| tipo:", x.dtype, f"| mín {x.min():.3f} | máx {x.max():.3f} | media {x.mean():.3f}")

augmented = SegDataset(DATA_DIR, [r], ("train", "val"), build_train_transform())
fig, axes = plt.subplots(1, 5, figsize=(13, 2.8))
for ax in axes:
    img, m = augmented[0]
    rgb = np.transpose(img, (1, 2, 0)) * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    ax.imshow(np.clip(rgb, 0, 1)); ax.axis("off")
plt.suptitle("Aumentos de una misma imagen"); plt.tight_layout(); plt.show()
print("Clases en la máscara tras aumentar:", sorted(set(np.unique(m).tolist())))

## 8. Entrenamiento y validación

Transfer learning: el encoder arranca con pesos de ImageNet y se congela en las 2 primeras épocas. Pérdida: entropía cruzada ponderada + Dice, ambas ignorando el 255. Se guarda el mejor modelo según el mIoU de validación y el entrenamiento para con *early stopping*.

**Esta celda reanuda sola.** Si existe un checkpoint en Drive, continúa desde la época siguiente, sin editar nada. Cada línea muestra el tiempo medido de la época y una estimación del tiempo restante.

In [ ]:
args = ["--data-root", DATA_DIR, "--out-dir", RUN_DIR, "--epochs", EPOCHS, "--batch-size", BATCH_SIZE,
        "--num-workers", NUM_WORKERS, "--patience", PATIENCE, "--manifest", "ml/splits/manifest.csv"]
if SMOKE_TEST:
    args += ["--limit-train", SMOKE_LIMITS["train"], "--limit-val", SMOKE_LIMITS["val"], "--device", "cpu"]
run(sys.executable, "ml/train.py", *args)

## 9. Curvas y análisis de sobreajuste/subajuste

El diagnóstico se calcula con el historial medido de esta corrida.

In [ ]:
history = json.loads((RUN_DIR / "history.json").read_text(encoding="utf-8"))
epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], "o-", label="train"); axes[0].plot(epochs, [h["val_loss"] for h in history], "o-", label="val")
axes[0].set_title("Pérdida"); axes[0].set_xlabel("época"); axes[0].legend()
axes[1].plot(epochs, [h["train_miou"] for h in history], "o-", label="train"); axes[1].plot(epochs, [h["val_miou"] for h in history], "o-", label="val")
axes[1].set_title("mIoU"); axes[1].set_xlabel("época"); axes[1].legend()
for i, name in enumerate(CLASSES):
    axes[2].plot(epochs, [h["val_iou_per_class"][name] for h in history], color=COLORS[i], label=name)
axes[2].set_title("IoU por clase en val"); axes[2].set_xlabel("época"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT_DIR / "curvas.png", dpi=120); plt.show()

best = max(history, key=lambda h: h["val_miou"])
last = history[-1]
gap_miou = last["train_miou"] - last["val_miou"]
gap_loss = last["val_loss"] - last["train_loss"]
best_val_loss = min(h["val_loss"] for h in history)
print(f"Épocas completadas: {len(history)} | tiempo total medido: {sum(h['seconds'] for h in history) / 60:.1f} min")
print(f"Mejor mIoU de val: {best['val_miou']:.4f} en la época {best['epoch']}")
print(f"Última época: mIoU train {last['train_miou']:.4f}, val {last['val_miou']:.4f} (brecha {gap_miou:+.4f}); pérdida train {last['train_loss']:.4f}, val {last['val_loss']:.4f} (brecha {gap_loss:+.4f})")
if SMOKE_TEST:
    diagnosis = "SMOKE_TEST: con tan pocas imágenes y épocas no se puede diagnosticar nada. Esto solo prueba el flujo."
elif gap_miou > 0.15 and last["val_loss"] > best_val_loss * 1.1:
    diagnosis = "Señal de SOBREAJUSTE: train sigue mejor que val y la pérdida de val se aleja de su mínimo."
elif last["train_miou"] < 0.5:
    diagnosis = "Señal de SUBAJUSTE: ni en train el mIoU es alto; el modelo o las épocas se quedan cortos."
else:
    diagnosis = "Sin señal clara de sobreajuste ni de subajuste con los umbrales usados (brecha > 0.15; mIoU de train < 0.5)."
print(diagnosis)

## 10. Exportación

`export.py` genera `kinetia_seg.pt`, `kinetia_seg.onnx` (opset 17, N dinámico, salida `logits` sin softmax), `manifest.json` y `hashes.sha256`, y verifica que `.pt` y `.onnx` coincidan con tolerancia `1e-4`. MiDaS se descarga solo para registrar su hash en `manifest.json`.

In [ ]:
midas_path = WORK / "midas" / "midas_small.onnx"
run(sys.executable, "ml/fetch_depth_model.py", "--out", midas_path)
run(sys.executable, "ml/export.py", "--checkpoint", RUN_DIR / "kinetia_seg.pt", "--midas", midas_path,
    "--out-dir", OUT_DIR, "--version", MODEL_VERSION, "--manifest", "ml/splits/manifest.csv")

## 11. Evaluación sobre el conjunto de prueba

Se evalúa el **`.onnx` exportado** con ONNX Runtime, no el `.pt`, para medir lo mismo que corre en producción. La partición de prueba solo se abre aquí. `metrics.json` es la única fuente de métricas para la API, la UI y el documento técnico.

In [ ]:
cmd = ["--models-dir", OUT_DIR, "--data-root", DATA_DIR, "--manifest", "ml/splits/manifest.csv"]
if SMOKE_TEST:
    cmd += ["--limit", SMOKE_LIMITS["test"], "--out", OUT_DIR / "metrics.json"]
run(sys.executable, "ml/evaluate.py", *cmd)

metrics = json.loads((OUT_DIR / "metrics.json").read_text(encoding="utf-8"))
matrix = np.array(metrics["confusion_matrix"], dtype=np.float64)
normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
im = axes[0].imshow(normalized, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(7), CLASSES, rotation=45, ha="right"); axes[0].set_yticks(range(7), CLASSES)
axes[0].set_xlabel("predicción"); axes[0].set_ylabel("verdad"); axes[0].set_title("Matriz de confusión (por fila)")
for i in range(7):
    for j in range(7):
        axes[0].text(j, i, f"{normalized[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if normalized[i, j] > 0.5 else "black")
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].barh(CLASSES[::-1], [metrics["per_class"][c]["iou"] for c in CLASSES[::-1]], color=COLORS[::-1])
axes[1].axvline(metrics["miou"], color="black", linestyle="--", label=f"mIoU = {metrics['miou']:.3f}")
axes[1].set_xlim(0, 1); axes[1].set_title("IoU por clase en prueba"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "evaluacion.png", dpi=120); plt.show()

## 12. Predicción sobre una imagen nueva, sin reentrenar

Carga el modelo exportado y predice una imagen que no está en el manifiesto. Para probar una foto propia, súbela al panel de archivos de Colab y pon su ruta en `NEW_IMAGE_PATH`.

In [ ]:
import urllib.request
import cv2
import onnxruntime as ort
from dataset import preprocess_seg

NEW_IMAGE_PATH = ""  # p. ej. "/content/mi_foto.jpg"; vacío usa una imagen de COCO test2017, que no tiene máscaras públicas
NEW_IMAGE_URL = "http://images.cocodataset.org/test2017/000000000016.jpg"
CONF_THRESHOLD = 0.5  # contrato, sección 6

if NEW_IMAGE_PATH:
    image_bgr = cv2.imread(NEW_IMAGE_PATH)
else:
    with urllib.request.urlopen(NEW_IMAGE_URL, timeout=60) as resp:
        image_bgr = cv2.imdecode(np.frombuffer(resp.read(), np.uint8), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("No se pudo leer la imagen.")

session = ort.InferenceSession(str(OUT_DIR / "kinetia_seg.onnx"), providers=["CPUExecutionProvider"])
t0 = time.time()
logits = session.run(["logits"], {"input": preprocess_seg(image_bgr)})[0][0]
inference_ms = (time.time() - t0) * 1000
exp = np.exp(logits - logits.max(axis=0, keepdims=True))
probs = exp / exp.sum(axis=0, keepdims=True)
pred = probs.argmax(axis=0)
pred[probs.max(axis=0) < CONF_THRESHOLD] = 0  # posproceso del contrato: confianza baja pasa a fondo
print(f"Inferencia medida en este equipo: {inference_ms:.0f} ms")
for i, name in enumerate(CLASSES):
    share = (pred == i).mean()
    if share > 0:
        print(f"  {name:<11} {share:7.2%} de los píxeles | confianza media {probs.max(axis=0)[pred == i].mean():.3f}")
rgb = cv2.cvtColor(cv2.resize(image_bgr, (320, 320), interpolation=cv2.INTER_LINEAR), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb); axes[0].set_title("Imagen nueva")
axes[1].imshow(palette[pred]); axes[1].set_title("Máscara predicha")
axes[2].imshow((0.55 * rgb + 0.45 * palette[pred]).astype(np.uint8)); axes[2].set_title("Superpuesta")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 13. Conclusiones

Esta celda redacta las conclusiones a partir de `metrics.json` y del historial. Ninguna cifra está escrita a mano.

In [ ]:
ranking = sorted(CLASSES, key=lambda c: metrics["per_class"][c]["iou"], reverse=True)
print(f"Sobre {metrics['num_images']} imágenes de prueba: mIoU {metrics['miou']:.4f}, Dice macro {metrics['dice_macro']:.4f}, exactitud de píxel {metrics['pixel_accuracy']:.4f}.")
print(f"Mejor clase: {ranking[0]} (IoU {metrics['per_class'][ranking[0]]['iou']:.4f}). Peor clase: {ranking[-1]} (IoU {metrics['per_class'][ranking[-1]]['iou']:.4f}).")
print("Objetivo del proyecto: mIoU ≥ 0.55 →", "se cumple." if metrics["miou"] >= 0.55 else "NO se cumple; se reporta la cifra real.")
print("Diagnóstico de la corrida:", diagnosis)
print("Limitaciones: fuego_luz mezcla la categoría light de COCO-Stuff con BoWFire, que tiene pocas imágenes y de incidentes; "
      "en esas imágenes el resto se ignora, así que allí no se miden falsos positivos de fuego.")
print("Noche y bosque no son clases: los deduce el motor con reglas (contrato, sección 1).")
if SMOKE_TEST:
    print("\nSMOKE_TEST: estas cifras NO son resultados del modelo; solo prueban que el flujo funciona.")

## 14. Descarga de los archivos

Crea `kinetia_artifacts.zip` con los 5 archivos, guarda una copia en Drive e imprime los hashes. Coloca el contenido en la carpeta `models/` del repositorio.

In [ ]:
import zipfile
from dataset import file_sha256
from validate_artifacts import ARTIFACTS, validate_dir

problems = validate_dir(OUT_DIR)
if problems:
    raise RuntimeError("Los archivos no cumplen el contrato: " + "; ".join(problems))
print("Los 5 archivos cumplen los esquemas del contrato.")

zip_path = OUT_DIR / "kinetia_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ARTIFACTS:
        zf.write(OUT_DIR / name, arcname=name)

final_dir = DRIVE_DIR / "artifacts"
final_dir.mkdir(parents=True, exist_ok=True)
for name in (*ARTIFACTS, "kinetia_artifacts.zip"):
    tmp = final_dir / (name + ".tmp")
    shutil.copyfile(OUT_DIR / name, tmp)
    os.replace(tmp, final_dir / name)

print(f"Copia en: {final_dir}")
print(f"SHA-256 kinetia_seg.pt:   {file_sha256(OUT_DIR / 'kinetia_seg.pt')}")
print(f"SHA-256 kinetia_seg.onnx: {file_sha256(OUT_DIR / 'kinetia_seg.onnx')}")
print(f"Tamaño del zip: {zip_path.stat().st_size / 1e6:.1f} MB")
if IN_COLAB and not SMOKE_TEST:
    from google.colab import files
    files.download(str(zip_path))